# §0 Setup & Configuration

This section establishes the **reproducible foundation** for the entire notebook.
Every subsequent phase (§2 catchment, §3 demographics, §4 competitors, §5 demand,
§6 financial model, §7 scenarios, §8 report) depends on the paths, keys, and
parameters defined here.

**v1 flaws addressed in this section:**
- *Hardcoded Google Drive paths* (v1 used literal Drive mount-point paths
  that only worked on the original author's Colab) — replaced with
  `PROJECT_ROOT`-relative `pathlib.Path`s that work on any machine.
- *Drive mount dependency* (v1 called the Drive mount API) — replaced with
  `git clone`, so caches travel with the repo and no manual Drive setup
  is needed.
- *Scattered constants* — consolidated into a single `BASE_ASSUMPTIONS` dict.

The **dual-environment pattern** detects Colab vs Windows at runtime and resolves
all paths accordingly. This is what makes "Restart & Run All" work in both
environments without manual intervention (PIPE-01, PIPE-02).

In [ ]:
# §0.1 — Installs & imports
# Only pip-install packages NOT preinstalled in Colab (RESEARCH.md §3).
# geopandas/shapely/pyproj/folium/pandas/matplotlib/jinja2 are preinstalled.
import os, sys, json, requests
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # requests-cache and python-dotenv are NOT preinstalled in Colab
    %pip install -q requests-cache python-dotenv

from requests_cache import CachedSession

print("[setup] imports loaded")

## §0.2 Environment Detection & Paths

v1 used hardcoded Google Drive mount-point paths that only worked on the
original author's Colab with a manually mounted Drive. This cell detects the
runtime environment and resolves `PROJECT_ROOT` via `pathlib` so the same code
works on Colab (after `git clone`) and on Windows.

**Key design decisions (from CONTEXT.md):**
- **D-06:** Colab clones the repo into `/content/medical-clinic`; caches come
  with the clone (committed to git per D-12).
- **D-07:** No Drive mount — explicitly rejected. Caches and code travel
  together via git.
- **D-12:** API response caches are committed to git, so a fresh clone can
  re-run the notebook offline/keyless at $0 (PIPE-04).

The API key loads from Colab Secrets (`userdata`) or local `.env`
(`python-dotenv`). A missing key degrades gracefully — cache hits satisfy
all fetches (PIPE-03).

In [ ]:
# §0.2 — Environment detection & path resolution (PIPE-02)
# TODO: replace <OWNER> with your GitHub username
if IN_COLAB:
    # Clone repo if not already present (caches come with it — D-06)
    import subprocess
    repo_path = "/content/medical-clinic"
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", "https://github.com/jatwell93/medical-clinic.git", repo_path],
                       check=True)
    PROJECT_ROOT = Path(repo_path)
    # Load API key from Colab Secrets
    try:
        from google.colab import userdata
        GOOGLE_PLACES_KEY = userdata.get("GOOGLE_PLACES_KEY")
    except Exception:
        GOOGLE_PLACES_KEY = ""
else:
    # Local Windows — load from .env via python-dotenv
    from dotenv import load_dotenv
    load_dotenv()  # Fails silently if .env missing — acceptable (cache fallback)
    GOOGLE_PLACES_KEY = os.environ.get("GOOGLE_PLACES_KEY", "")
    PROJECT_ROOT = Path.cwd()  # Notebooks don't have __file__

# Graceful degradation: key is optional if cache is populated (D-12)
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
assert CACHE_DIR.exists() or GOOGLE_PLACES_KEY, \
    "Need GOOGLE_PLACES_KEY or a populated data/cache/ to run."

print(f"[env] IN_COLAB={IN_COLAB}, PROJECT_ROOT={PROJECT_ROOT}")
print(f"[env] Google Places key: {'loaded' if GOOGLE_PLACES_KEY else 'empty (cache-only mode)'}")

## §0.3 Parameters & Assumptions

This is the **single source of truth** for every numeric assumption in the
notebook (PIPE-05). v1 had constants scattered across cells — making it
impossible to audit or run scenarios without hunting through the notebook.

**Why a flat dict?** The flat-dict shape makes Phase 5 scenarios trivial:
`{**BASE_ASSUMPTIONS, **overrides}`. A `@dataclass` would break this pattern;
an external YAML/JSON file would break the "single parameters cell" wording.

**Rule (from ARCHITECTURE.md):** any numeric literal in §2–§8 that isn't a
unit conversion is a bug. Downstream cells *read* these values; they never
redefine them.

Every value has an inline comment with a **source citation** or an
`# UNCONFIRMED` flag so the assumptions register (Phase 5, REP-01) can be
assembled automatically from this cell.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  SINGLE PARAMETERS CELL — every assumption lives here (PIPE-05)
#  Downstream cells READ these; they never redefine them.
#  Scenarios (Phase 5): {**BASE_ASSUMPTIONS, **overrides}
# ═══════════════════════════════════════════════════════════════

SITE_ADDRESS = "292-296 Johnston St, Abbotsford VIC 3067"
CATCHMENT_RADII_M = [1000, 3000, 5000]  # 1/3/5 km rings
PEER_POSTCODES = ["3067", "3066", "3068", "3070", "3078", "3079",
                  "3101", "3121", "3122", "3123"]

# Cache control — single global refresh flag (D-04)
FORCE_REFRESH = False

BASE_ASSUMPTIONS = {
    # --- Site & catchment ---
    "site_address":           SITE_ADDRESS,
    "catchment_radii_m":      CATCHMENT_RADII_M,
    "peer_postcodes":         PEER_POSTCODES,

    # --- Demand (source: MBS SA3 20604 Yarra; RACGP GP:pop benchmarks) ---
    "consults_per_capita_yr": {"0-24": 4.5, "25-44": 5.5, "45-64": 7.5, "65+": 12.0},  # AIHW (2026) fallback
    "consults_per_gp_day":    28,          # RACGP typical full-book GP

    # --- Revenue (source: MBS item 23 rebate; local gap-fee scan) ---
    "bulk_bill_share":        0.70,        # DECISION: mixed-billing model (70% bulk / 30% private)
    "std_rebate":             43.90,       # MBS item 23, 2025-26 — verify at build time
    "private_fee":            95.00,       # inner-Melb typical standard consult — UNCONFIRMED
    "gp_revenue_share":       0.35,        # % of billings retained by clinic (GPs keep 65%) — UNCONFIRMED range 30-35%

    # --- Costs ---
    "rent_yr":                100_000,     # UNCONFIRMED — key sensitivity, flagged in report
    "fitout_capital":         350_000,     # estimate — flagged, sensitivity-tested
    "n_gp_fte":               5,
    "n_allied_fte":           1,

    # --- Operational ---
    "days_per_yr":            220,         # GP working days after leave
    "utilisation":            0.75,        # ramp-up target utilisation

    # --- Phase 2: Catchment (source: ABS ASGS Edition 3; PITFALLS.md Pitfall 1/2) ---
    "sa1_shapefile":          "SA1_2021_AUST_GDA2020.zip",  # ABS ASGS Edition 3 — manual download
    "poa_shapefile":          "POA_2021_AUST_GDA2020_SHP.zip",  # ABS ASGS Edition 3 — on hand
    "assert_3km_buffer_km2":  28.27,       # π×3² — sanity check for EPSG:7855 buffering (GEO-02, D-09a)
    "assert_3km_buffer_tol":  0.1,         # km² tolerance
    "catchment_pop_plausible_range": (80_000, 110_000),  # inner-Melbourne 3km ring (D-09b, PITFALLS.md Pitfall 2)
    "erp_vintage_year":       2024,        # ABS Regional Population 2023-24 FY (released March 2025) — DEMO-04

    # --- Phase 3: Demand & Competitors (source: AIHW 2026, MBS SA3, RACGP, AMWAC) ---
    # AIHW SA3 age bands: 0-24, 25-44, 45-64, 65+ (NOT 0-14/15-64/65+ — Correction 2)
    # Values are "services per 100 people" from AIHW (2026), SA3 20604 Yarra
    # These are placeholder defaults — replaced by load_aihw_sa3_rates() at runtime
    "consults_per_capita_yr": {"0-24": 4.5, "25-44": 5.5, "45-64": 7.5, "65+": 12.0},  # AIHW (2026) fallback
    "mbs_sa3_filename":       "medicare-quarterly-statistics-statistical-area-sa3-summary-march-quarter-2025-26.xlsx",  # D-01 manual download
    "aihw_age_band_filename": "aihw-medicare-subsidised-gp-allied-health-2024-25-data-tables.xlsx",  # D-02 manual download
    "avg_fte_per_clinic":     4.0,        # RACGP 2019 (5.7 headcount) × DoH 2024 (0.74 FTE/GP) = 4.2, rounded down — D-10
    "amwac_per_100k":         110.4,      # AMWAC 2000 planning benchmark (1:905) — D-10; latest actuals: 113 nat, 117 VIC
    "gp_fte_consults_per_yr": 5500,       # ~110-130 consults/week × ~46 weeks — midpoint for 5 FTE = 27,500/yr — D-13
    "consults_per_patient_yr": 5.0,       # patients-needed framing denominator — D-13c (AIHW nat avg 6.8, regular patients ~5)
    "mbs_item_23_rebate":     43.90,      # MBS Online, from 1 Jul 2025 (2.4% indexation) — confirmed
    "market_share_thresholds": {"low": 5, "high": 15},  # <5% low, 5-15% moderate, >15% high — D-14
    "places_included_types":  ["doctor", "pharmacy", "physiotherapist", "dentist", "medical_lab"],  # Places API (New) Table A — D-05
}

print(f"[params] {len(BASE_ASSUMPTIONS)} assumptions loaded")
print(f"[params] FORCE_REFRESH={FORCE_REFRESH}")

# §1 Data Acquisition & Caching

v1 had **no caching** — every re-run hit the ABS and Google APIs, costing
money and making the notebook non-reproducible offline. This section
establishes a single `CachedSession` that **all** external calls go through
(ABS Data API, Google Geocoding, Google Places).

**What this gives us (PIPE-04):**
- Re-runs are **free** — cached responses are served from disk.
- Re-runs are **offline-capable** — no network needed if cache is populated.
- Re-runs are **keyless** — ABS needs no key; Places/Geocode cache hits
  need no key. A grader or investor can clone the repo and run the entire
  notebook without any API keys.

The ABS smoke test below proves the cache works using a **keyless** endpoint
(the dataflow list requires no API key). On first run it fetches from the
network; on second run it serves from cache (`response.from_cache == True`).

In [ ]:
# §1.1 — Cache session construction (D-01, D-02, D-03)
# Single cached HTTP boundary for ALL external calls (D-01, D-02, D-03)
# requests-cache filesystem backend: one JSON file per response in data/cache/
# allowable_methods includes POST for Google Places API (New) — STACK.md requirement
CACHE_DIR.mkdir(parents=True, exist_ok=True)

session = CachedSession(
    cache_name=str(CACHE_DIR / "api_cache"),
    backend='filesystem',
    allowable_methods=('GET', 'POST'),  # POST needed for Places API (New)
    expire_after=-1,                     # No expiry — census is immutable vintage
)

# Apply FORCE_REFRESH flag from PARAMS cell (D-04)
if FORCE_REFRESH:
    session.cache.clear()
    print("[cache] cleared (FORCE_REFRESH=True)")

print(f"[cache] session ready → {CACHE_DIR}")

## §1.2 ABS Data API Smoke Test

This proves the cache layer works using a **keyless** ABS endpoint. The
dataflow list (`/rest/dataflow?detail=allstubs`) requires no API key, so
this test runs even in cache-only mode (no `GOOGLE_PLACES_KEY`).

**Important format note:** the ABS REST API returns **SDMX-ML XML** by
default (Content-Type: `application/vnd.sdmx.structure+xml`), not JSON.
This is the standard SDMX format for statistical data exchange. The actual
census data fetch in Phase 2 will use `?format=csvfilewithlabels` to get
human-readable CSV with both dimension codes and labels — but this smoke
test only validates that the cache layer works, so we accept the XML
structure response and verify it looks like SDMX.

**What it validates (PIPE-04):**
- The `CachedSession` is correctly constructed and can make HTTP requests.
- On first run: fetches from the network and writes a cache file.
- On second run: serves from cache (`response.from_cache == True`).
- Cache files appear in `data/cache/`.

This is the foundation that makes every subsequent API call (ABS census,
Google geocode, Google Places) free and offline-reproducible.

In [ ]:
# §1.2 — ABS Data API smoke test (PIPE-04 validation)
# Keyless smoke test — proves the cache layer works (PIPE-04)
# ABS dataflow list endpoint requires no API key.
# NOTE: ABS REST API returns SDMX-ML XML by default (Content-Type:
# application/vnd.sdmx.structure+xml), NOT JSON. The data pipeline (Phase 2)
# will use ?format=csvfilewithlabels for actual census data; this smoke test
# only validates the cache layer, so we accept the XML structure response.
ABS_DATAFLOW_URL = "https://data.api.abs.gov.au/rest/dataflow?detail=allstubs"

response = session.get(ABS_DATAFLOW_URL)
print(f"[abs] HTTP {response.status_code}, from_cache={response.from_cache}")
print(f"[abs] Content-Type: {response.headers.get('Content-Type', 'unknown')}")

# Validate response — ABS returns SDMX-ML XML, not JSON
assert response.status_code == 200, f"ABS returned HTTP {response.status_code}"
content_type = response.headers.get("Content-Type", "")
assert "sdmx" in content_type or "xml" in content_type, \
    f"Unexpected Content-Type: {content_type}"
assert len(response.content) > 0, "Empty response body"
# Sanity check: SDMX structure messages contain the sdmx namespace
assert b"sdmx" in response.content[:2000], \
    "Response body does not look like SDMX-ML"

# Cache validation assertions (RESEARCH.md Validation Architecture)
# requests-cache filesystem backend stores responses in a subdirectory
# named after cache_name (e.g. data/cache/api_cache/*.json), so use rglob.
assert CACHE_DIR.exists(), "Cache directory not created"
cache_files = list(CACHE_DIR.rglob("*.json"))
assert len(cache_files) > 0, "No cache files created — cache layer not working"

print(f"[abs] {len(cache_files)} cache file(s) in {CACHE_DIR}")
print(f"[abs] Smoke test PASSED — cache layer operational")

## §1.2 Geocode Site

v1 used the **postcode centroid** as the catchment centre (PITFALLS.md Pitfall 3)
and referenced the site coordinates before they were defined (out-of-order
execution). This cell geocodes the **exact address** ONCE through the cached
session, prints the coordinates, and asks the reader to visually verify the pin
on the immediately-following map before proceeding.

The cached response means re-runs are **free and keyless** — after the first
successful geocode, the coordinates are served from `data/cache/` and no
`GOOGLE_PLACES_KEY` is needed (PIPE-04).

**Known anchor (D-31):** The site is at the Johnston St × Hoddle St corner,
Abbotsford. Expected coordinates ≈ (-37.799, 145.003). The map below lets you
confirm the pin lands on the right spot before any buffer or Places search
centres on it.

In [ ]:
# Geocode the exact site address through the Phase 1 CachedSession (D-31, PITFALLS.md Pitfall 3)
# v1 flaw: used postcode centroid + referenced site coords before definition (out-of-order execution)
import urllib.parse

GEOCODE_URL = (
    "https://maps.googleapis.com/maps/api/geocode/json?address="
    + urllib.parse.quote(SITE_ADDRESS)
    + f"&key={GOOGLE_PLACES_KEY}"
)

# Cached fetch — re-runs are free and keyless after first run (PIPE-04)
if GOOGLE_PLACES_KEY or any(CACHE_DIR.glob("geocode_*.json")):
    resp = session.get(GEOCODE_URL)
    data = resp.json()
    assert data.get("status") == "OK", f"Geocode failed: {data.get('status')} {data.get('error_message','')}"
    result = data["results"][0]
    site_lat = result["geometry"]["location"]["lat"]
    site_lon = result["geometry"]["location"]["lng"]
    site_place_id = result.get("place_id", "")
    print(f"[geocode] {SITE_ADDRESS}")
    print(f"[geocode] lat={site_lat:.6f}, lon={site_lon:.6f} (from_cache={resp.from_cache})")
    print(f"[geocode] ⚠ Visually verify this pin on the map below before proceeding.")
    print(f"[geocode]    Expected: Johnston St × Hoddle St corner, Abbotsford")
else:
    raise RuntimeError(
        "No GOOGLE_PLACES_KEY and no cached geocode response. "
        "Add the key via Colab Secrets or local .env, run once, then re-run keyless."
    )

In [ ]:
# §1.2 verification map — visually confirm the pin (D-31)
import folium
m_verify = folium.Map(location=[site_lat, site_lon], zoom_start=16, tiles="CartoDB Positron")
folium.Marker(
    [site_lat, site_lon],
    popup=f"Site: {SITE_ADDRESS}<br>({site_lat:.5f}, {site_lon:.5f})",
    tooltip="292-296 Johnston St",
    icon=folium.Icon(color="red", icon="info-sign"),
).add_to(m_verify)
m_verify

# §2 Geospatial Catchment

v1's headline flaw was **whole-postcode summing** (PITFALLS.md Pitfall 2):
it summed the FULL population of every postcode touching the buffer, inflating
catchment population 2–4× and flipping the go/no-go answer. This section fixes
it with **SA1-level area apportionment** (D-01).

All metric operations (buffer, area, distance) happen in **EPSG:7855**
(GDA2020 / MGA zone 55) — PITFALLS.md Pitfall 1 warns that buffering in
degrees creates a "buffer" spanning the continent. The CRS convention is:
**metric ops in EPSG:7855, display in EPSG:4326**.

The v1-vs-v2 comparison at the end (§2.4) is the teaching moment showing
v1 inflated 2–4×.

In [ ]:
# Load SA1 + POA boundary geometry from data/local/ (D-02, D-07)
# Both are gitignored — large, immutable, not committed
# Geospatial imports (preinstalled in Colab — STACK.md)
import geopandas as gpd
from shapely.geometry import Point

SA1_ZIP = PROJECT_ROOT / "data" / "local" / BASE_ASSUMPTIONS["sa1_shapefile"]
POA_ZIP = PROJECT_ROOT / "data" / "local" / BASE_ASSUMPTIONS["poa_shapefile"]

if not SA1_ZIP.exists():
    print(f"⚠ SA1 shapefile missing: {SA1_ZIP}")
    print(f"  Download from ABS ASGS Edition 3 digital boundary files (see .env.example)")
    print(f"  Falling back to POA-level apportionment (degraded — uniform-density assumption coarser)")
    USE_SA1 = False
else:
    USE_SA1 = True

poa = gpd.read_file(POA_ZIP)
poa = poa[poa["STE_NAME21"] == "Victoria"].to_crs("EPSG:7855")  # metric CRS for area math
print(f"[geom] POA: {len(poa)} VIC polygons, CRS={poa.crs.to_epsg()}")

if USE_SA1:
    sa1 = gpd.read_file(SA1_ZIP)
    sa1 = sa1[sa1["STE_NAME21"] == "Victoria"].to_crs("EPSG:7855")
    print(f"[geom] SA1: {len(sa1)} VIC polygons, CRS={sa1.crs.to_epsg()}")

## §2.2 Build 1/3/5 km Buffers (EPSG:7855)

The CRS convention is **metric ops in EPSG:7855, display in EPSG:4326**
(PITFALLS.md Pitfall 1). The 3 km buffer area assertion (≈ 28.27 km² = π×3²)
is the sanity check that catches a degree-based buffer before it poisons
everything downstream.

v1 buffered in degrees → the "buffer" spanned half of Victoria. The assertion
below would have caught it instantly.

In [ ]:
# Build 1/3/5 km buffers in EPSG:7855 (GEO-02, D-09a, PITFALLS.md Pitfall 1)
# v1 flaw: buffered in degrees → "buffer" spanning half of Victoria
site_point = gpd.GeoDataFrame(
    [{"name": "site"}], geometry=[Point(site_lon, site_lat)], crs="EPSG:4326"
)
site_metric = site_point.to_crs("EPSG:7855")

buffers = {}
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    buf = site_metric.buffer(r)
    buffers[r] = buf
    area_km2 = buf.area.iloc[0] / 1e6
    print(f"[buffer] {r//1000} km: area={area_km2:.2f} km²")

# Sanity assertion (GEO-02, D-09a) — 3km buffer ≈ 28.27 km² (π×3²)
area_3k = buffers[3000].area.iloc[0] / 1e6
assert abs(area_3k - BASE_ASSUMPTIONS["assert_3km_buffer_km2"]) < BASE_ASSUMPTIONS["assert_3km_buffer_tol"], \
    f"3km buffer area {area_3k:.2f} km² ≠ {BASE_ASSUMPTIONS['assert_3km_buffer_km2']} km² — CRS bug (PITFALLS.md Pitfall 1)"
print(f"[buffer] ✓ 3km buffer area assertion passed ({area_3k:.2f} km² ≈ π×3²)")

## §2.3 SA1 Area Apportionment

This is the **headline v1 fix** (PITFALLS.md Pitfall 2). v1 summed the FULL
population of every postcode touching the buffer. v2 area-apportions each
SA1's population by the fraction of its area inside the buffer.

The **uniform-density assumption** is documented (D-05) and the Yarra River
is annotated on every map so readers see the geographic issue — the river and
industrial land in Abbotsford mean the assumption likely *overstates*
population in that slice.

**Note:** The SA1 total persons come from §3 (Plan 02-02) via
`fetch_sa1_total_persons(sa1_codes)`. This cell defines the apportionment
logic and spatial filter; the population weights are wired in §3 once the ABS
SA1 census is fetched.

In [ ]:
# SA1-level area apportionment (D-01 — the headline v1 fix, PITFALLS.md Pitfall 2)
# v1 flaw: summed FULL population of every postcode touching the buffer (2-4× overstatement)
# v2: area-weight each SA1's population by the fraction inside the buffer
if USE_SA1:
    # Spatial-filter SA1s intersecting the 5km buffer (D-08 — beat 30s API timeout)
    sa1_in_5k = sa1[sa1.geometry.intersects(buffers[5000].iloc[0])].copy()
    print(f"[apportion] {len(sa1_in_5k)} SA1s intersect 5km buffer")

    # Per-ring area-weighted apportionment
    # sa1_pop_df is produced in §3 (Plan 02-02) — stub here, wired there
    def apportion_ring(sa1_gdf, sa1_pop_df, ring_geom):
        """Area-weight SA1 populations into a ring. Returns total population in ring."""
        ring = gpd.GeoDataFrame(geometry=[ring_geom], crs="EPSG:7855")
        inter = sa1_gdf.overlay(ring, how="intersection")
        # Join population
        inter = inter.merge(sa1_pop_df, on="SA1_CODE21", how="left")
        # Area fraction (intersect_area / full_sa1_area)
        full_area = sa1_gdf.set_index("SA1_CODE21")["geometry"].area
        inter["frac"] = inter.geometry.area / inter["SA1_CODE21"].map(full_area).values
        inter["pop_in_ring"] = inter["frac"] * inter["Total_P_P"]
        return inter["pop_in_ring"].sum(), inter

    # NOTE: sa1_pop_df is assigned in §3 demographics (Plan 02-02).
    # For Phase 2 §2 standalone validation, we use a proxy: SA1 area-only weights
    # (uniform density across all SA1s in 5km). The real population weights land in §3.
    # This stub is overwritten by §3 once ABS SA1 census is fetched.
    print("[apportion] ℹ Population weights come from §3 (ABS C21_G01_SA1). "
          "Catchment population totals are computed in §3 after the SA1 census fetch.")
else:
    print("[apportion] ⚠ SA1 shapefile missing — using POA-level apportionment (degraded)")
    # POA-level fallback (coarser) — same area-weight logic at POA granularity

## §2.4 v1-vs-v2 Catchment Comparison

This is the **teaching moment** (D-06, success criterion #2). v1's naive
whole-postcode sum is reproduced inline so readers see the flawed logic next
to the fixed logic. The grouped bar chart + side-by-side table show v1
inflating 2–4× per ring.

The v1 and v2 totals come from §3 (Plan 02-02). This cell defines the comparison
function `compare_v1_v2(v1_ring_pops, v2_ring_pops)` that §3.3 calls after
computing both the v1 naive whole-postcode sums and the v2 apportioned totals.

In [ ]:
# v1-vs-v2 catchment comparison (D-06, success criterion #2)
# Reproduce v1's naive whole-postcode sum inline, compare against v2 apportioned
def v1_naive_catchment_pop(ring_geom_m, poa_pop_df):
    """v1 flaw: sum FULL POA population for any POA touching the buffer.
    poa_pop_df: DataFrame with columns POA_CODE21, Total_P_P (from §3 G01 fetch).
    Returns the summed population (int), NOT a GeoDataFrame."""
    touching = poa[poa.geometry.intersects(ring_geom_m)]
    # Join POA total persons onto the touching POAs and sum (v1's naive approach)
    merged = touching.merge(poa_pop_df[["POA_CODE21", "Total_P_P"]], on="POA_CODE21", how="left")
    return int(merged["Total_P_P"].sum())

def compare_v1_v2(v1_ring_pops: dict, v2_ring_pops: dict):
    """
    v1_ring_pops: {radius_m: naive_whole_postcode_pop} — v1's flawed approach
    v2_ring_pops: {radius_m: sa1_apportioned_pop} — v2's corrected approach
    Returns comparison DataFrame + renders grouped bar chart showing v1 overstatement.
    """
    import pandas as pd
    import matplotlib.pyplot as plt
    rows = []
    for r, v2_pop in v2_ring_pops.items():
        v1_pop = v1_ring_pops.get(r, 0)
        rows.append({"ring_km": r//1000, "v1_naive": v1_pop, "v2_apportioned": v2_pop,
                     "diff": v1_pop - v2_pop,
                     "pct_overstate": (v1_pop/v2_pop - 1)*100 if v2_pop else None})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    x = range(len(df))
    ax.bar([i-0.2 for i in x], df["v1_naive"], width=0.4, label="v1 naive (whole-postcode)", color="#d62728")
    ax.bar([i+0.2 for i in x], df["v2_apportioned"], width=0.4, label="v2 apportioned (SA1)", color="#2ca02c")
    ax.set_xticks(list(x)); ax.set_xticklabels([f"{r} km" for r in df["ring_km"]])
    ax.set_ylabel("Catchment population"); ax.legend(); ax.set_title("v1 vs v2 catchment population")
    plt.show()
    return df
print("[compare] compare_v1_v2(v1_ring_pops, v2_ring_pops) defined — called in §3.3 after both v1 and v2 totals are computed")

## §2.5 Catchment Maps

**Folium** is interactive (inline only, NOT in the PDF — it's HTML/JS and
won't render in weasyprint, STACK.md constraint). The **static matplotlib +
contextily** twin handles the PDF.

One static figure per ring (1 km, 3 km, 5 km), each zoomed to its ring (D-27).
The Yarra River is annotated on every map so readers see the uniform-density
caveat's geographic issue (D-05).

In [ ]:
# Folium interactive map — inline only, NOT in PDF (D-25, D-30, STACK.md)
import folium
m_catch = folium.Map(location=[site_lat, site_lon], zoom_start=12, tiles="CartoDB Positron")

# Site pin
folium.Marker([site_lat, site_lon], popup=SITE_ADDRESS,
              icon=folium.Icon(color="red", icon="info-sign")).add_to(m_catch)

# 1/3/5 km buffer rings (reproject to EPSG:4326 for folium)
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    buf_4326 = gpd.GeoDataFrame(geometry=buffers[r], crs="EPSG:7855").to_crs("EPSG:4326")
    folium.GeoJson(buf_4326, style_function=lambda f, r=r: {
        "color": {1000:"blue", 3000:"orange", 5000:"green"}[r],
        "weight": 2, "fillOpacity": 0.05
    }, name=f"{r//1000} km ring").add_to(m_catch)

# POA boundaries (peers) — shaded by apportionment share (computed in §3)
poa_peers_4326 = poa[poa["POA_CODE21"].isin(PEER_POSTCODES)].to_crs("EPSG:4326")
folium.GeoJson(poa_peers_4326, style_function=lambda f: {"color":"purple","weight":1,"fillOpacity":0.1},
               name="Peer POAs").add_to(m_catch)

# Yarra River line (D-05) — approximate path through Abbotsford
# NOTE: exact Yarra geometry comes from OSM or a local GeoJSON; for MVP, a hand-drawn
# polyline through known points is acceptable. Plan 02-02 may upgrade to OSM fetch.
yarra_pts = [[-37.808, 145.003], [-37.810, 145.000], [-37.815, 144.998], [-37.820, 144.995]]
folium.PolyLine(yarra_pts, color="aqua", weight=4, opacity=0.7, tooltip="Yarra River (approx)").add_to(m_catch)

folium.LayerControl().add_to(m_catch)
print("[map] Folium catchment map rendered (inline only — PDF uses static matplotlib below)")
m_catch

In [ ]:
# Static matplotlib + contextily figures for the PDF (D-27, D-29) — one per ring
import matplotlib.pyplot as plt
import contextily as cx
import numpy as np

site_4326 = site_point.to_crs("EPSG:4326")
poa_peers_4326 = poa[poa["POA_CODE21"].isin(PEER_POSTCODES)].to_crs("EPSG:4326")

for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    fig, ax = plt.subplots(figsize=(8, 8))
    # Buffer ring
    buf_4326 = gpd.GeoDataFrame(geometry=buffers[r], crs="EPSG:7855").to_crs("EPSG:4326")
    buf_4326.boundary.plot(ax=ax, color={1000:"blue", 3000:"orange", 5000:"green"}[r], linewidth=2)
    # POA boundaries
    poa_peers_4326.boundary.plot(ax=ax, color="purple", linewidth=0.8, alpha=0.6)
    # Site pin
    site_4326.plot(ax=ax, color="red", markersize=40)
    # Yarra River (approx)
    yarra_x = [145.003, 145.000, 144.998, 144.995]
    yarra_y = [-37.808, -37.810, -37.815, -37.820]
    ax.plot(yarra_x, yarra_y, color="aqua", linewidth=3, alpha=0.7, label="Yarra River (approx)")
    # Basemap
    cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)
    ax.set_title(f"Catchment — {r//1000} km ring (Johnston St site)")
    ax.legend(loc="upper right")
    plt.show()

## §2.6 Uniform-Density Caveat

> **Caveat (uniform-density assumption):** Catchment population is
> area-apportioned at SA1 level assuming population is uniformly distributed
> within each SA1. This likely *overstates* population in the Yarra River /
> industrial slice of Abbotsford (non-residential land). The Yarra River is
> annotated on every catchment map so readers can see the geographic issue.
> Mesh-block apportionment (finer) is the rigorous alternative — noted in
> PITFALLS.md Pitfall 2 as the "best" option; deferred as overkill for an
> MVP investor report.

# §3 Demographics

v1 had **no census staleness handling** (PITFALLS.md Pitfall 15) and
**conflated POA/SA2/SA3 geographies** (Pitfall 5). This section fetches
ABS G01/G02/G04 at POA level for peer benchmarking + SA1 level for
catchment apportionment, with a **local GCP fallback** that emits the SAME
tidy schema (D-17) so downstream code is source-agnostic.

**ERP scaling** (2021 → 2024) addresses the 4-5 year census lag — Abbotsford
has grown since the 2021 Census. The peer benchmarking table includes
**placeholder GP/pharmacy columns** ("— Phase 3") that Phase 3 fills —
no Places API calls in Phase 2.

Critically, §3 also **wires the SA1 total persons into the §2.3
apportionment function** — without this, the v2 catchment population totals
and the v1-vs-v2 comparison are stubs. This section completes the headline
v1-flaw-fix teaching moment.

In [ ]:
# ABS Data API — G01/G02 for POA 3067 + 9 peers (DEMO-01, D-13, D-14)
# All calls through the Phase 1 CachedSession (ARCHITECTURE.md Pattern 1)
import pandas as pd
import io

ABS_BASE = "https://data.api.abs.gov.au/rest"
PEER_POA_CODES = "+".join(PEER_POSTCODES)  # SDMX OR operator within a dimension

def tidy_abs_csv(df):
    """Ensure ABS CSV is in wide format (one row per geography, measures as columns).
    ABS Data API csvfilewithlabels can return long format (one row per observation
    with MEASURE + OBS_VALUE columns) — pivot to wide if detected (WR-09 fix).
    If already wide (measures are columns), return unchanged."""
    if "OBS_VALUE" in df.columns and "MEASURE" in df.columns:
        # Long format detected — pivot to wide
        index_cols = [c for c in df.columns if c not in ("MEASURE", "OBS_VALUE")]
        if not index_cols:
            print("⚠ tidy_abs_csv: long format detected but no index columns — returning as-is")
            return df
        print(f"⚠ tidy_abs_csv: long format detected — pivoting on {index_cols}")
        df = df.pivot_table(index=index_cols, columns="MEASURE", values="OBS_VALUE", aggfunc="first").reset_index()
        df.columns.name = None  # remove the MEASURE axis name
    return df

def fetch_abs_csv(flow_id, data_key):
    """Fetch ABS dataflow as CSV-with-labels through the cached session. Returns tidy (wide) DataFrame."""
    url = f"{ABS_BASE}/data/{flow_id}/{data_key}?format=csvfilewithlabels"
    resp = session.get(url)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    # ABS API may return long format (MEASURE + OBS_VALUE columns) — pivot to wide (WR-09 fix)
    df = tidy_abs_csv(df)
    return df, resp

def check_dataflow_exists(flow_id):
    """Check if a dataflow ID exists in the ABS dataflow list (D-11).
    Safe default: returns True on any failure so the fetch is always attempted —
    downstream functions have their own try/except fallbacks (no-hard-fail principle).
    The dataflow endpoint returns SDMX-ML XML, not JSON, so .json() would crash (CR-01 fix)."""
    try:
        flows_resp = session.get(f"{ABS_BASE}/dataflow?detail=allstubs")
        # Endpoint returns SDMX-ML XML (verified by §1.2 smoke test) — parse with ElementTree
        import xml.etree.ElementTree as ET
        root = ET.fromstring(flows_resp.text)
        # SDMX-ML namespace for dataflow structures
        ns = {"str": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure"}
        flow_ids = [f.get("id", "") for f in root.findall(".//str:Dataflow", ns)]
        if not flow_ids:
            # Namespace might differ — try without namespace
            flow_ids = [f.get("id", "") for f in root.iter() if "Dataflow" in f.tag]
        if not flow_ids:
            # Valid XML but not SDMX-ML (e.g. CDN/maintenance error page) —
            # can't determine dataflow existence; return True to attempt fetch
            # (no-hard-fail principle, WR-05 fix)
            print(f"⚠ check_dataflow_exists({flow_id}): XML parsed but no Dataflow elements — defaulting to True (will attempt fetch)")
            return True
        return flow_id in flow_ids
    except Exception as e:
        print(f"⚠ check_dataflow_exists({flow_id}) failed: {e} — defaulting to True (will attempt fetch)")
        return True

def parse_gcp_g01_to_tidy(xlsx_path):
    """Parse local GCP POA 3067 xlsx into the same schema as the API path (D-17).
    Schema parity: column names, dtypes, row-per-POA shape must match API path.
    Returns empty DataFrame with correct schema if xlsx parsing is not yet implemented —
    zero is not a valid population (WR-02 fix)."""
    print(f"⚠ parse_gcp_g01_to_tidy: xlsx cell parsing not yet implemented — returning empty schema")
    print(f"  GCP fallback values are not populated; peer table will show N/A for 3067 (WR-02)")
    df = pd.DataFrame(columns=["POA_CODE21", "Total_P_P", "M_P", "F_P", "Total_Dwll_D"])
    return df

def fetch_g01_poa():
    """G01: total persons, sex split, dwelling counts (D-13)."""
    try:
        # dataKey dimension order is runtime-verified via the datastructure endpoint
        # Pattern: {MEASURE}+{codes}.{POA}+{codes}.{FREQ}.{TIME}
        df, resp = fetch_abs_csv("ABS,C21_G01_POA,latest", f"all.{PEER_POA_CODES}.A.2021")
        print(f"[abs] G01 POA: {len(df)} rows from API (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ABS G01 POA API failed: {e}")
        print(f"  Falling back to local GCP_POA3067.xlsx (D-15, D-16)")
        print(f"  Peer comparison degraded — 9 peer rows marked N/A")
        # Local fallback — parse GCP xlsx into the SAME tidy schema (D-17)
        xlsx_path = PROJECT_ROOT / "data" / "local" / "GCP_POA3067.xlsx"
        if xlsx_path.exists():
            df = parse_gcp_g01_to_tidy(xlsx_path)
            return df, "fallback"
        else:
            print(f"⚠ GCP fallback file also missing: {xlsx_path}")
            # Empty-schema DataFrame (consistent with fallback case, WR-08 fix) —
            # zero is not a valid population; columns allow safe downstream merge
            return pd.DataFrame(columns=["POA_CODE21", "Total_P_P", "M_P", "F_P", "Total_Dwll_D"]), "none"

def fetch_g02_poa():
    """G02: median age, median income (D-14)."""
    try:
        df, resp = fetch_abs_csv("ABS,C21_G02_POA,latest", f"all.{PEER_POA_CODES}.A.2021")
        print(f"[abs] G02 POA: {len(df)} rows from API (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ABS G02 POA API failed: {e}")
        print(f"  Falling back to local GCP_POA3067.xlsx (D-15, D-16)")
        print(f"  Peer comparison degraded — 9 peer rows marked N/A")
        xlsx_path = PROJECT_ROOT / "data" / "local" / "GCP_POA3067.xlsx"
        if xlsx_path.exists():
            # Parse G02 medians from GCP xlsx — schema parity with API path (D-17, WR-02)
            print(f"⚠ G02 GCP fallback: xlsx cell parsing not yet implemented — returning empty schema (WR-02)")
            df = pd.DataFrame(columns=["POA_CODE21", "Median_age_persons", "Median_tot_hshld_inc_weekly"])
            return df, "fallback"
        else:
            print(f"⚠ GCP fallback file also missing: {xlsx_path}")
            # Empty-schema DataFrame (consistent with fallback case, WR-08 fix)
            return pd.DataFrame(columns=["POA_CODE21", "Median_age_persons", "Median_tot_hshld_inc_weekly"]), "none"

g01_df, g01_source = fetch_g01_poa()
print(f"[abs] G01 source: {g01_source}, shape: {g01_df.shape}")

g02_df, g02_source = fetch_g02_poa()
print(f"[abs] G02 source: {g02_source}, shape: {g02_df.shape}")

In [ ]:
# G04 age by sex — runtime-verify C21_G04_POA existence (D-11, D-12)
# v1 had no age-band data — this feeds the Phase 3 demand model (age-band attendance rates)
# Age bands use ABS standard 5-year structure: 0-4, 5-9, ..., 80-84, 85+ (D-18)
G04_POA_EXISTS = check_dataflow_exists("C21_G04_POA")
print(f"[abs] C21_G04_POA exists: {G04_POA_EXISTS}")

def fetch_g04_poa():
    """G04: age by sex in ABS standard 5-year bands (D-18)."""
    try:
        df, resp = fetch_abs_csv("ABS,C21_G04_POA,latest", f"all.{PEER_POA_CODES}.A.2021")
        print(f"[abs] G04 POA: {len(df)} rows from API (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ABS G04 POA API failed: {e}")
        print(f"  Falling back to G04 fallback (D-12) — consistent with G01/G02 fallback (WR-03 fix)")
        return fetch_g04_fallback()

def fetch_g04_fallback():
    """G04 fallback if C21_G04_POA not available at POA level (D-12).
    Fallback: C21_G04_SA2 + apportion, or local GCP for 3067 65+ share with peers N/A."""
    print("⚠ C21_G04_POA not available — using fallback (D-12)")
    print("  Fallback: local GCP for 3067 65+ share, peers marked N/A")
    # Return a stub DataFrame with pct_65plus for 3067 only
    df = pd.DataFrame([{
        "POA_CODE21": "3067",
        "pct_65plus": float("nan"),  # NaN not 0 — zero is not a valid 65+ share (WR-07 fix)
    }])
    return df, "fallback"

if G04_POA_EXISTS:
    g04_df, g04_source = fetch_g04_poa()
    # Derive 65+ share: sum bands >= 65 / total (D-18 — ABS standard 5-year bands)
    # Age bands: 0-4, 5-9, 10-14, ..., 60-64, 65-69, 70-74, 75-79, 80-84, 85+
    # Use case-insensitive regex matching to cover all known ABS G04 column variants (WR-06 fix):
    #   Age_yr_65_69, Age_yr_85ov (API CSV-with-labels)
    #   Age_65_69, Age_85plus (DataPack exact)
    #   Age_85_over, Age_65_69_yr (DataPack _over/_yr suffix variants)
    #   AGE_YR_65_69, AGE_85_OVER (uppercase variants)
    import re
    _age_band_re = re.compile(r"age(?:_yr)?_(65|70|75|80|85)", re.IGNORECASE)
    age_bands_65plus = [b for b in g04_df.columns if _age_band_re.match(b)]
    print(f"[abs] G04 65+ age bands matched: {age_bands_65plus}")
    if not age_bands_65plus:
        # Fallback: print all columns for debugging if no bands matched (teaching transparency)
        print(f"⚠ No 65+ age bands matched — G04 columns: {list(g04_df.columns)}")
        print(f"  Check ABS G04 data dictionary for exact column names (WR-04)")
    if age_bands_65plus and "Total_P_P" in g04_df.columns:
        g04_df["pct_65plus"] = g04_df[age_bands_65plus].sum(axis=1) / g04_df["Total_P_P"] * 100
    else:
        print("⚠ Could not derive pct_65plus from G04 columns — check schema")
        g04_df["pct_65plus"] = 0
else:
    g04_df, g04_source = fetch_g04_fallback()

print(f"[abs] G04 source: {g04_source}, shape: {g04_df.shape}")

In [ ]:
# SA1 total persons — for the §2.3 apportionment weights (D-03, D-08)
# Spatial-filter SA1s intersecting 5km buffer first (beat 30s timeout — PITFALLS.md Pitfall 4)
# This wires the SA1 population into the apportionment function to compute
# actual v2 catchment population totals — the headline v1-flaw fix completion.
if USE_SA1:
    sa1_codes = sa1_in_5k["SA1_CODE21"].tolist()
    print(f"[abs] Fetching SA1 total persons for {len(sa1_codes)} SA1s")

    SA1_G01_EXISTS = check_dataflow_exists("C21_G01_SA1")
    print(f"[abs] C21_G01_SA1 exists: {SA1_G01_EXISTS}")

    if SA1_G01_EXISTS:
        # Fetch C21_G01_SA1 for only the spatial-filtered SA1 IDs (D-08)
        sa1_codes_or = "+".join(sa1_codes)
        try:
            sa1_pop_df, sa1_resp = fetch_abs_csv("ABS,C21_G01_SA1,latest", f"all.{sa1_codes_or}.A.2021")
            sa1_pop_df = sa1_pop_df[["SA1_CODE21", "Total_P_P"]]  # tidy schema
            sa1_source = "api"
            print(f"[abs] SA1 total persons: {len(sa1_pop_df)} rows, source={sa1_source} (from_cache={sa1_resp.from_cache})")
        except Exception as e:
            print(f"⚠ ABS C21_G01_SA1 API failed: {e}")
            print(f"  Falling back to SA1 GCP DataPack (D-03)")
            sa1_datapack = PROJECT_ROOT / "data" / "local" / "2021Census_G01_AUST_SA1.csv"
            if sa1_datapack.exists():
                sa1_pop_df = pd.read_csv(sa1_datapack)
                sa1_pop_df = sa1_pop_df[sa1_pop_df["SA1_CODE21"].isin(sa1_codes)][["SA1_CODE21", "Total_P_P"]]
                sa1_source = "datapack"
                print(f"[abs] SA1 total persons: {len(sa1_pop_df)} rows, source={sa1_source}")
            else:
                print(f"⚠ SA1 DataPack missing: {sa1_datapack}")
                print(f"  Download from ABS Census DataPacks (SA1 GCP) — see .env.example")
                sa1_pop_df = None
                sa1_source = "none"
    else:
        print("⚠ C21_G01_SA1 not available — falling back to SA1 GCP DataPack (D-03)")
        sa1_datapack = PROJECT_ROOT / "data" / "local" / "2021Census_G01_AUST_SA1.csv"
        if sa1_datapack.exists():
            sa1_pop_df = pd.read_csv(sa1_datapack)
            sa1_pop_df = sa1_pop_df[sa1_pop_df["SA1_CODE21"].isin(sa1_codes)][["SA1_CODE21", "Total_P_P"]]
            sa1_source = "datapack"
            print(f"[abs] SA1 total persons: {len(sa1_pop_df)} rows, source={sa1_source}")
        else:
            print(f"⚠ SA1 DataPack missing: {sa1_datapack}")
            print(f"  Download from ABS Census DataPacks (SA1 GCP) — see .env.example")
            sa1_pop_df = None
            sa1_source = "none"

    if sa1_pop_df is not None:
        # Wire into §2.3 apportionment — compute actual v2 catchment population totals
        v2_ring_pops = {}
        v1_ring_pops = {}
        for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
            pop, inter = apportion_ring(sa1_in_5k, sa1_pop_df, buffers[r].iloc[0])
            v2_ring_pops[r] = pop
            print(f"[catchment] {r//1000} km ring: v2 apportioned pop = {pop:,.0f}")

            # v1 naive: sum FULL POA population for any POA touching the buffer (CR-02 fix)
            # Uses g01_df from §3.1 — the POA total persons already fetched
            if g01_df is not None and len(g01_df) > 0 and "POA_CODE21" in g01_df.columns:
                v1_pop = v1_naive_catchment_pop(buffers[r].iloc[0], g01_df)
                v1_ring_pops[r] = v1_pop
                print(f"[catchment] {r//1000} km ring: v1 naive pop = {v1_pop:,.0f} (whole-postcode sum)")
            else:
                print(f"⚠ g01_df empty — v1 naive pop unavailable for {r//1000} km ring")
                v1_ring_pops[r] = 0

        # Plausibility assertion (D-09b, PITFALLS.md Pitfall 2)
        lo, hi = BASE_ASSUMPTIONS["catchment_pop_plausible_range"]
        assert lo < v2_ring_pops[3000] < hi, \
            f"3km catchment pop {v2_ring_pops[3000]:,.0f} outside plausible range ({lo:,}-{hi:,}) — PITFALLS.md Pitfall 2"
        print(f"[catchment] ✓ 3km pop plausibility assertion passed ({v2_ring_pops[3000]:,.0f})")

        # Call the §2.4 v1-vs-v2 comparison with BOTH v1 and v2 totals (D-06 completion, CR-02 fix)
        if v1_ring_pops:
            comparison_df = compare_v1_v2(v1_ring_pops, v2_ring_pops)
        else:
            print("[catchment] ⚠ v1 naive pops unavailable — v1-vs-v2 comparison skipped")
    else:
        print("[catchment] ⚠ SA1 pop unavailable — v2 totals + v1 comparison deferred")
        v2_ring_pops = {}
        v1_ring_pops = {}
else:
    print("[catchment] ⚠ SA1 shapefile missing — v2 totals deferred (POA-level fallback)")
    v2_ring_pops = {}
    v1_ring_pops = {}

## §3.4 ERP Scaling (Census Staleness)

The 2021 Census is ~4-5 years old (PITFALLS.md Pitfall 15). Abbotsford has
grown. **ERP (Estimated Resident Population)** is the ABS's official annual
update. We scale 2021 census → 2024 ERP-adjusted using the Yarra SA2 growth
rate (D-21 — uniform rate is defensible since the catchment is largely within
Yarra SA2).

The caveat is printed and both raw + scaled values are shown side-by-side
(D-22). ERP is fetched through the same CachedSession (D-23).

ERP scaling is applied to BOTH the catchment population AND the peer
benchmarking table (D-24) — consistent treatment.

In [ ]:
# ERP scaling — 2021 census → 2024 ERP-adjusted (DEMO-04, D-19..D-24, PITFALLS.md Pitfall 15)
# Source: ABS_ANNUAL_ERP_ASGS2021 (annual ERP by state and sub-state geography)
ERP_VINTAGE = BASE_ASSUMPTIONS["erp_vintage_year"]  # 2024 (2023-24 FY, released March 2025)

def fetch_erp_sa2():
    """Fetch ERP by SA2 from ABS_ANNUAL_ERP_ASGS2021. Returns DataFrame with SA2 code, year, population."""
    try:
        # dataKey: {MEASURE}.{REGION}+{SA2_codes}.{FREQ}.{TIME}
        # Yarra SA2 code — verify at runtime via datastructure introspection
        # ASGS Edition 3 Yarra SA2 code: 206041001 (verify at runtime)
        df, resp = fetch_abs_csv("ABS,ABS_ANNUAL_ERP_ASGS2021,latest", f"all.206041001.A.2021+2024")
        print(f"[erp] SA2 ERP fetched: {len(df)} rows (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ERP API failed: {e} — ERP scaling skipped (DEMO-04 degraded)")
        return None, "none"

erp_df, erp_source = fetch_erp_sa2()
erp_growth_rate = 1.0  # default: no scaling if ERP unavailable

if erp_df is not None:
    # Compute Yarra SA2 growth rate 2021 → 2024
    pop_2021 = erp_df[erp_df["TIME_PERIOD"]==2021]["OBS_VALUE"].iloc[0]
    pop_2024 = erp_df[erp_df["TIME_PERIOD"]==2024]["OBS_VALUE"].iloc[0]
    erp_growth_rate = pop_2024 / pop_2021
    print(f"[erp] Yarra SA2: 2021={pop_2021:,.0f} → 2024={pop_2024:,.0f} (rate={erp_growth_rate:.4f})")

    # Apply to catchment population (D-24)
    if v2_ring_pops:
        v2_ring_pops_erp = {r: pop * erp_growth_rate for r, pop in v2_ring_pops.items()}
        print(f"[erp] Catchment 3km: 2021={v2_ring_pops[3000]:,.0f} → 2024 ERP={v2_ring_pops_erp[3000]:,.0f}")
    else:
        v2_ring_pops_erp = {}
        print("[erp] Catchment pop not available — ERP scaling on catchment deferred")

    # Caveat (D-22) — printed markdown note
    print(f"\n[erp] ⚠ Caveat: Population scaled from 2021 Census baseline to {ERP_VINTAGE} ERP-adjusted")
    print(f"  Method: ABS Regional Population (3218.0) Yarra SA2 growth rate applied (D-21)")
    print(f"  Round aggressively — false precision is misleading (PITFALLS.md Pitfall 15)")
else:
    v2_ring_pops_erp = {}
    print("[erp] ⚠ ERP unavailable — peer table uses unscaled 2021 census values")

## §3.5 Peer Benchmarking Table

The peer set frames the market context (DEMO-02, DEMO-03). Census columns
now: population (raw 2021 + ERP-scaled), median age, 65+ share, median
income. GP count + pharmacy count are **placeholders** ("— Phase 3") for
Phase 3 to fill (D-26). No Places API calls in Phase 2.

The 10 peer postcodes are already in `BASE_ASSUMPTIONS["peer_postcodes"]`
from Phase 1. This table merges G01/G02/G04 into a single tidy table keyed
by POA.

In [ ]:
# Peer benchmarking table (DEMO-02, DEMO-03, D-26)
# Census columns now; GP/pharmacy columns are placeholders for Phase 3
peer_table = g01_df[["POA_CODE21", "Total_P_P"]].merge(
    g02_df[["POA_CODE21", "Median_age_persons", "Median_tot_hshld_inc_weekly"]],
    on="POA_CODE21", how="outer"
).merge(
    g04_df[["POA_CODE21", "pct_65plus"]],
    on="POA_CODE21", how="outer"
)

# ERP-scaled population column (D-24)
if erp_df is not None:
    peer_table["Total_P_P_erp"] = (peer_table["Total_P_P"] * erp_growth_rate).round(0)
else:
    peer_table["Total_P_P_erp"] = peer_table["Total_P_P"]  # unscaled fallback

# Placeholder columns for Phase 3 (D-26)
peer_table["gp_count"] = "— Phase 3"
peer_table["pharmacy_count"] = "— Phase 3"

# Highlight 3067 (site POA)
peer_table["is_site"] = peer_table["POA_CODE21"] == "3067"

print(peer_table.to_string(index=False))

## §3.6 Peer Comparison Charts

2×2 bar chart grid (D-29): population (ERP-scaled), median age, 65+ share,
median income. POA 3067 highlighted in each chart. This is the visual
peer-context frame for the investor report.

In [ ]:
# Peer comparison 2×2 bar chart grid (D-29)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
charts = [
    ("Total_P_P_erp", "Population (ERP-scaled)"),
    ("Median_age_persons", "Median age"),
    ("pct_65plus", "65+ share (%)"),
    ("Median_tot_hshld_inc_weekly", "Median weekly household income ($)"),
]
for ax, (col, title) in zip(axes.flat, charts):
    colors = ["#d62728" if s else "#2ca02c" for s in peer_table["is_site"]]
    ax.bar(peer_table["POA_CODE21"], peer_table[col], color=colors)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Peer postcode comparison — POA 3067 (site) highlighted", y=1.02)
plt.tight_layout()
plt.show()

## §1.4 Places API (New) Client

v1 used the **legacy Places Nearby Search** (`/maps/api/place/nearbysearch/json`)
with `next_page_token` pagination and uncached `requests.get` (Pitfall 10).
The legacy API is closed to new customers since March 2025. This cell uses
**Places API (New)** `POST places:searchNearby` with a mandatory
`X-Goog-FieldMask` header (STACK.md).

**Key differences from legacy (Pitfall 8):** Nearby Search (New) has NO
pagination — `maxResultCount` caps at 20. The saturation subdivision (D-05)
beats this by splitting saturated circles into a 2×2 grid of smaller circles
and deduping on `place.id`.

**Caching:** all calls route through the Phase 1 `CachedSession` with
`allowable_methods=('GET','POST')` so re-runs are $0. The field mask uses
**Pro SKU fields ONLY** — `rating` and `userRatingCount` are Enterprise SKU
(Correction 1) and are dropped to stay in the $32/1k Pro tier with 5,000
free/month.

In [ ]:
# §1.4 Places API (New) Client — Nearby Search with saturation subdivision (D-05)
# Replaces v1's legacy google_nearby_places (uncached requests.get, next_page_token pagination)
# Places API (New): POST places:searchNearby, mandatory X-Goog-FieldMask, maxResultCount=20, NO pagination
import math

PLACES_URL = "https://places.googleapis.com/v1/places:searchNearby"

# Pro SKU field mask ONLY — drops rating/userRatingCount (Enterprise SKU, Correction 1)
# Pro SKU: $32/1k, 5,000 free/month. Enterprise: $35/1k, 1,000 free/month.
PLACES_FIELD_MASK = (
    "places.id,"
    "places.displayName,"
    "places.location,"
    "places.types,"
    "places.primaryType,"
    "places.primaryTypeDisplayName,"
    "places.businessStatus,"
    "places.formattedAddress,"
    "places.shortFormattedAddress,"
    "places.addressComponents"
)

def places_nearby(lat, lon, radius_m, included_types):
    """Single Nearby Search (New) call. Returns list of place dicts. Max 20."""
    body = {
        "includedTypes": included_types,
        "maxResultCount": 20,
        "locationRestriction": {
            "circle": {
                "center": {"latitude": lat, "longitude": lon},
                "radius": float(radius_m)
            }
        },
        "languageCode": "en-AU"
    }
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_PLACES_KEY,
        "X-Goog-FieldMask": PLACES_FIELD_MASK
    }
    resp = session.post(PLACES_URL, json=body, headers=headers)  # Phase 1 CachedSession
    resp.raise_for_status()
    return resp.json().get("places", [])

def offset_meters(lat, lon, dx_m, dy_m):
    """Offset lat/lon by dx/dy meters. Good enough for <10km in Melbourne."""
    dlat = dy_m / 111_320
    dlon = dx_m / (111_320 * abs(math.cos(math.radians(lat))))
    return lat + dlat, lon + dlon

def places_nearby_saturated(lat, lon, radius_m, included_types):
    """Query per-type × per-ring; auto-subdivide 2×2 grid on 20-result cap (D-05)."""
    results = places_nearby(lat, lon, radius_m, included_types)
    if len(results) < 20:
        return results  # Not saturated

    # Saturated → subdivide into 2×2 grid of smaller circles
    # Sub-circle radius = original_radius / 2 (covers the diameter, overlaps at edges)
    sub_radius = radius_m / 2
    offsets = [
        (sub_radius * 0.5, sub_radius * 0.5),    # NE
        (sub_radius * 0.5, -sub_radius * 0.5),   # SE
        (-sub_radius * 0.5, sub_radius * 0.5),   # NW
        (-sub_radius * 0.5, -sub_radius * 0.5),  # SW
    ]
    sub_results = []
    sub_queries = []
    for dx, dy in offsets:
        sub_lat, sub_lon = offset_meters(lat, lon, dx, dy)
        r = places_nearby(sub_lat, sub_lon, sub_radius, included_types)
        sub_queries.append((sub_lat, sub_lon, r))
        sub_results.extend(r)

    # If any sub-circle also saturates at 20, sub-subdivide that quarter once (max depth 2)
    for sub_lat, sub_lon, r in sub_queries:
        if len(r) >= 20:
            for sdx, sdy in offsets:
                ss_lat, ss_lon = offset_meters(sub_lat, sub_lon, sdx * 0.5, sdy * 0.5)
                sub_results.extend(places_nearby(ss_lat, ss_lon, sub_radius / 2, included_types))

    # Dedupe on place.id
    seen = {}
    for p in results + sub_results:
        pid = p.get("id")
        if pid and pid not in seen:
            seen[pid] = p
    return list(seen.values())

print("[places] places_nearby_saturated() defined — Pro SKU field mask, 2×2 subdivision on 20-cap, dedupe on place.id")

In [ ]:
# Query site catchment: 5 types × 3 radii = 15 base requests (D-05)
site_places = {}
for ptype in BASE_ASSUMPTIONS["places_included_types"]:
    for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
        results = places_nearby_saturated(site_lat, site_lon, r, [ptype])
        site_places[(ptype, r)] = results
        print(f"[places] {ptype} {r//1000}km: {len(results)} places (after dedupe)")
total_site = sum(len(v) for v in site_places.values())
print(f"[places] site catchment total: {total_site} place records (pre cross-type dedupe)")

In [ ]:
# Peer catchment queries (D-06) — 10 peers × 3 radii × 5 types = ~150 requests (cached)
peer_places = {}
for poa_code in PEER_POSTCODES:
    poa_row = poa[poa["POA_CODE21"] == poa_code]
    if poa_row.empty:
        print(f"[places] ⚠ POA {poa_code} not found in geometry — skipping")
        continue
    # centroid is in EPSG:7855; convert to lat/lon for Places API
    centroid_4326 = poa_row.to_crs("EPSG:4326").geometry.centroid.iloc[0]
    peer_lat, peer_lon = centroid_4326.y, centroid_4326.x
    for ptype in BASE_ASSUMPTIONS["places_included_types"]:
        for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
            results = places_nearby_saturated(peer_lat, peer_lon, r, [ptype])
            peer_places[(poa_code, ptype, r)] = results
print(f"[places] peer queries complete: {len(peer_places)} (type, radius) pairs across {len(PEER_POSTCODES)} peers")

## §1.5 MBS SA3 Data + AIHW Age-Band Rates

v1 loaded the **state-level** Medicare file
(`medicare-quarterly-statistics-primary-care-service-type-summary...xlsx`)
whose sheets are `['Contents', 'State', 'Modified Monash', 'Primary Care
Service Types']` — **NO SA3 data** (Pitfall 6). Any "local utilisation"
from it is just the VIC average in disguise.

This cell loads the **correct product**: "Medicare quarterly statistics –
Statistical Area (SA3) Summary" from health.gov.au, filtered to SA3 20604
Yarra (D-01). The AIHW age-band rates provide per-capita GP attendance by
age band — the SA3 age bands are **0-24, 25-44, 45-64, 65+** (NOT
0-14/15-64/65+ — Correction 2). The AIHW latest edition is **2026** covering
2024-25 data (Correction 3).

The state file remains as a **fallback** with a LOUD warning (D-04). Both
datasets are manual downloads to `data/local/` (like the SA1 shapefile in
Phase 2).

In [ ]:
# §1.5 MBS SA3 loader — SA3 20604 Yarra with state fallback (D-01, D-04, Pitfall 6)
# MBS SA3 Summary (March quarter 2025-26), published 25 May 2026. health.gov.au
import pandas as pd

DATA_LOCAL_DIR = PROJECT_ROOT / "data" / "local"
MBS_SA3_FILENAME = BASE_ASSUMPTIONS.get("mbs_sa3_filename",
    "medicare-quarterly-statistics-statistical-area-sa3-summary-march-quarter-2025-26.xlsx")

def load_mbs_sa3():
    """Load SA3-level MBS GP Non-Referred Attendance data. Fallback to state with warning (D-04)."""
    sa3_path = DATA_LOCAL_DIR / MBS_SA3_FILENAME

    if sa3_path.exists():
        xl = pd.ExcelFile(sa3_path)
        print(f"[mbs] SA3 file sheets: {xl.sheet_names}")
        # Find the SA3 sheet — try "SA3" first, fall back to first sheet containing "SA3"
        sa3_sheet = None
        for name in xl.sheet_names:
            if "SA3" in str(name).upper():
                sa3_sheet = name
                break
        if sa3_sheet is None:
            sa3_sheet = xl.sheet_names[0]
            print(f"[mbs] ⚠ No 'SA3' sheet found, using first sheet: {sa3_sheet}")
        df = pd.read_excel(sa3_path, sheet_name=sa3_sheet)
        # Filter to SA3 20604 Yarra
        sa3_yarra = df[df["SA3"].astype(str) == "20604"]
        if sa3_yarra.empty:
            print("⚠ WARNING: SA3 20604 not found in file. Check SA3 column name.")
        print(f"[mbs] SA3 20604 Yarra: {len(sa3_yarra)} quarters loaded")
        return sa3_yarra, "sa3"
    else:
        # D-04: State fallback with LOUD warning
        state_path = DATA_LOCAL_DIR / "medicare-quarterly-statistics-primary-care-service-type-summary-march-quarter-2025-26(1).xlsx"
        print("═" * 70)
        print("⚠  STATE BENCHMARK, NOT LOCAL — SA3 20604 data not found.")
        print("⚠  Download the SA3 Summary file from health.gov.au")
        print("⚠  and place in data/local/. Demand signal = VIC average, not Yarra.")
        print("═" * 70)
        df = pd.read_excel(state_path, sheet_name="State")
        vic = df[df["State"] == "VIC"]
        return vic, "state_fallback"

In [ ]:
# §1.5 AIHW age-band rates loader (D-02, Correction 2, Correction 3)
# AIHW (2026) — "Medicare-subsidised GP, allied health and specialist health care across local areas"
# Updated 26 Mar 2026, covers 2017-18 to 2024-25. SA3 age bands: 0-24, 25-44, 45-64, 65+
AIHW_FILENAME = BASE_ASSUMPTIONS.get("aihw_age_band_filename",
    "aihw-medicare-subsidised-gp-allied-health-2024-25-data-tables.xlsx")

def load_aihw_sa3_rates():
    """Load AIHW SA3-level GP attendance rates by age band.
    Returns (dict, source_str) for SA3 20604 Yarra."""
    path = DATA_LOCAL_DIR / AIHW_FILENAME
    if not path.exists():
        print("⚠ WARNING: AIHW age-band data not found. Using national average fallback.")
        return ({"0-24": 4.5, "25-44": 5.5, "45-64": 7.5, "65+": 12.0}, "national_fallback")

    xl = pd.ExcelFile(path)
    print(f"[aihw] sheets: {xl.sheet_names}")

    # Find the SA3 GP attendances sheet — verify at runtime
    sa3_sheet = None
    for name in xl.sheet_names:
        if "SA3" in str(name).upper() and "GP" in str(name).upper():
            sa3_sheet = name
            break
    if sa3_sheet is None:
        # Fall back to first sheet containing "SA3"
        for name in xl.sheet_names:
            if "SA3" in str(name).upper():
                sa3_sheet = name
                break
    if sa3_sheet is None:
        sa3_sheet = xl.sheet_names[0]
        print(f"[aihw] ⚠ No SA3 sheet found, using first sheet: {sa3_sheet}")

    df = pd.read_excel(path, sheet_name=sa3_sheet)
    # Filter to SA3 20604
    sa3_yarra = df[df["SA3"].astype(str) == "20604"]

    # Extract rates by age band (column structure verified at runtime)
    rates = {}
    for band in ["0-24", "25-44", "45-64", "65+"]:
        # Column name pattern verified at runtime — may be "0-24" or "0_24" etc.
        col = [c for c in df.columns if band.replace("-", "") in str(c).replace("-", "")]
        if col:
            rates[band] = float(sa3_yarra[col[0]].iloc[0])
        else:
            print(f"⚠ Age band '{band}' column not found in AIHW data")

    return (rates, "aihw_sa3")

In [ ]:
# Load MBS SA3 + AIHW age-band rates (D-01, D-02, D-03)
mbs_sa3_df, mbs_source = load_mbs_sa3()
aihw_rates, aihw_source = load_aihw_sa3_rates()
print(f"[mbs] source: {mbs_source}, rows: {len(mbs_sa3_df)}")
print(f"[aihw] source: {aihw_source}, rates: {aihw_rates}")
# Assert SA3 20604 present when using SA3 data (Pitfall 6 verification)
if mbs_source == "sa3":
    assert mbs_sa3_df["SA3"].astype(str).str.contains("20604").any(), \
        "SA3 20604 not found in MBS data — check file (Pitfall 6)"
    print("[mbs] ✓ SA3 20604 Yarra confirmed in data")

# §4 Competitor Landscape

v1's raw `type=doctor` Google Places counts were **inflated 2–5×** by
three problems (Pitfall 9):
1. **Specialists** — skin clinics, cosmetic clinics, and vets mis-tagged
   as "doctor" in Google Places
2. **Individual practitioner listings** — each GP gets their own Places
   entry *plus* the practice entry (double-counting)
3. **Miscategorised places** — radiology, pathology, optometry flagged as
   "medical" but not GP clinics

v1 concluded "6.6 doctors per 1,000" when the sane metro benchmark is
~1.2 FTE GPs per 1,000. This section fixes it with:
- **Keyword-based classification** (D-09) — pharmacy brands first, then
  exclude keywords, corporate GP brands, independent GP indicators,
  allied health indicators
- **rapidfuzz fuzzy-dedupe** (D-09) of individual-practitioner listings
  (MIT-licensed, NOT the deprecated GPL alternative)
- **Manual-review checkpoint** (D-11) for ambiguous rows — printed inline,
  no blocking prompts
- **GeoJSON persistence** (D-08) — deduped GeoDataFrame saved to
  `data/cache/competitors.geojson` so re-runs never touch the API (Pitfall 10)

The per-ring competitor counts feed the GP FTE capacity estimate in §5.
The peer competitor table (D-16) fills the DEMO-03 placeholder with real data.

In [ ]:
# §4.1 Classification rules (D-09) — keyword-based competitor bucketing
# pip install rapidfuzz  # MIT-licensed fuzzy string matching
# Checks pharmacy brands FIRST (D-07), then exclude keywords, corporate GP
# brands, independent GP indicators, allied health indicators, else ambiguous.

# Corporate GP brands (from gpzoo.com.au 2025 — verified site counts)
CORPORATE_GP_BRANDS = {
    "ipn": "IPN (Sonic Healthcare)",
    "myhealth": "Amplar Health (Myhealth/Medibank)",
    "family doctor": "Family Doctor",
    "forhealth": "ForHealth",
    "healius": "ForHealth (formerly Healius)",
    "bupa": "Bupa (Partnered Health)",
    "better medical": "Amplar Health (Better Medical)",
    "jupiter health": "Jupiter Health",
    "ochre": "Ochre Health",
    "sonic": "Sonic Healthcare",
    "qualitas": "Qualitas",
    "health&co": "ForHealth (Health&Co)",
}

# Pharmacy brands (D-07) — keyword rules on displayName
PHARMACY_BRANDS = {
    "chemist warehouse": "Chemist Warehouse",
    "priceline": "Priceline",
    "amcal": "Amcal",
    "terrywhite": "TerryWhite Chemmart",
    "terry white": "TerryWhite Chemmart",
    "chemmart": "TerryWhite Chemmart",
    "sigma": "Sigma (API)",
    "pharmacy 4 less": "Pharmacy 4 Less",
    "national pharmacy": "National Pharmacy",
    "soul pattinson": "Soul Pattinson",
    "discount drug stores": "Discount Drug Stores",
}

# Exclude keywords — these are NOT GP clinics (21 terms)
EXCLUDE_KEYWORDS = [
    "skin", "cosmetic", "dermatology", "laser", "beauty",
    "vet", "veterinary", "animal",
    "optometry", "optometrist", "podiatry", "audiology",
    "radiology", "imaging", "pathology", "laboratory",
    "specialist", "paediatric", "obstetric", "gynaecology", "psychiatry",
]

def classify_place(name, primary_type, types_list):
    """Classify a place into: gp_corporate, gp_independent, pharmacy, allied_health, exclude, ambiguous."""
    name_lower = str(name).lower()

    # 1. Check pharmacy brands first (most specific — D-07)
    for brand, label in PHARMACY_BRANDS.items():
        if brand in name_lower:
            return ("pharmacy", label)

    if primary_type == "pharmacy" or "pharmacy" in (types_list or []):
        return ("pharmacy", "Independent")

    # 2. Check exclude bucket (specialists, cosmetic, vet, etc.)
    for kw in EXCLUDE_KEYWORDS:
        if kw in name_lower:
            return ("exclude", kw)

    # 3. Check corporate GP brands
    for brand, label in CORPORATE_GP_BRANDS.items():
        if brand in name_lower:
            return ("gp_corporate", label)

    # 4. Check independent GP clinic indicators
    gp_clinic_indicators = ["medical centre", "medical center", "general practice",
                            "gp clinic", "gp practice", "medical clinic", "family practice",
                            "health centre", "health center", "community health"]
    if any(kw in name_lower for kw in gp_clinic_indicators):
        return ("gp_independent", "Independent")

    # 5. doctor-type without clinic indicator → ambiguous (manual review)
    if primary_type in ("doctor", "medical_center", "medical_clinic"):
        return ("ambiguous", "doctor-type but no clinic indicator in name")

    # 6. Allied health indicators
    allied_indicators = ["physio", "physiotherapy", "psychology", "psychologist",
                         "dental", "dentist", "chiropractic", "chiropractor",
                         "occupational therapy", "dietitian", "dietician",
                         "speech", "acupuncture", "osteopath"]
    if any(kw in name_lower for kw in allied_indicators):
        return ("allied_health", "Allied Health")
    if primary_type in ("physiotherapist", "dentist", "dental_clinic", "chiropractor"):
        return ("allied_health", "Allied Health")

    # 7. Unclassified → ambiguous
    return ("ambiguous", f"Unclassified (primaryType={primary_type})")

print("[classify] classify_place() defined — 12 corporate GP brands, 11 pharmacy brands, 21 exclude keywords")

In [ ]:
# §4.2 Fuzzy-dedupe + GeoDataFrame construction + GeoJSON persistence (D-08, D-09)
# rapidfuzz: MIT-licensed, C++-fast, drop-in fuzz.token_sort_ratio API
from rapidfuzz import fuzz
import geopandas as gpd
from shapely.geometry import Point

def fuzzy_dedupe_competitors(places_gdf, name_threshold=85, address_threshold=80):
    """Dedupe individual-practitioner listings that share a practice address.
    Uses rapidfuzz.token_sort_ratio on normalised name + formattedAddress.
    Keeps the first occurrence (by place.id), merges duplicates into a list."""
    keep = []
    seen = []
    duplicates = []

    for idx, row in places_gdf.iterrows():
        name_norm = str(row.get("displayName", "")).lower().strip()
        addr_norm = str(row.get("formattedAddress", "")).lower().strip()
        is_dup = False

        for kept in seen:
            name_sim = fuzz.token_sort_ratio(name_norm, kept["name"])
            addr_sim = fuzz.token_sort_ratio(addr_norm, kept["addr"])
            # Match if name is very similar AND address is very similar
            if name_sim >= name_threshold and addr_sim >= address_threshold:
                is_dup = True
                duplicates.append({
                    "kept_id": kept["id"],
                    "dup_id": row.get("id"),
                    "name": row.get("displayName"),
                    "name_sim": name_sim,
                    "addr_sim": addr_sim,
                })
                break

        if not is_dup:
            keep.append(idx)
            seen.append({"id": row.get("id"), "name": name_norm, "addr": addr_norm})

    deduped = places_gdf.loc[keep].copy()
    return deduped, duplicates

# Build GeoDataFrame from site_places raw results
# Flatten all (type, radius) results into a list of place dicts
all_places = []
for (ptype, radius), places in site_places.items():
    for p in places:
        pid = p.get("id", "")
        dn = p.get("displayName", {})
        name = dn.get("text", "") if isinstance(dn, dict) else str(dn)
        loc = p.get("location", {})
        lat = loc.get("latitude", None) if isinstance(loc, dict) else None
        lon = loc.get("longitude", None) if isinstance(loc, dict) else None
        ptdn = p.get("primaryTypeDisplayName", {})
        ptdn_text = ptdn.get("text", "") if isinstance(ptdn, dict) else str(ptdn)
        record = {
            "id": pid,
            "displayName": name,
            "lat": lat,
            "lon": lon,
            "types": p.get("types", []),
            "primaryType": p.get("primaryType", ""),
            "primaryTypeDisplayName": ptdn_text,
            "businessStatus": p.get("businessStatus", ""),
            "formattedAddress": p.get("formattedAddress", ""),
            "query_type": ptype,
            "query_radius": radius,
        }
        all_places.append(record)

# Dedupe on place.id first (cross-type duplicates from saturation subdivision)
seen_ids = {}
unique_places = []
for rec in all_places:
    if rec["id"] and rec["id"] not in seen_ids:
        seen_ids[rec["id"]] = True
        unique_places.append(rec)
    elif not rec["id"]:
        unique_places.append(rec)  # keep records without id

# Classify each place
for rec in unique_places:
    category, label = classify_place(rec["displayName"], rec["primaryType"], rec["types"])
    rec["category"] = category
    rec["label"] = label

# Build GeoDataFrame
geometry = [Point(r["lon"], r["lat"]) for r in unique_places if r["lon"] is not None and r["lat"] is not None]
valid_records = [r for r in unique_places if r["lon"] is not None and r["lat"] is not None]
competitors_gdf = gpd.GeoDataFrame(valid_records, geometry=geometry, crs="EPSG:4326")

# Fuzzy-dedupe individual-practitioner listings (D-09)
competitors_gdf, dup_list = fuzzy_dedupe_competitors(competitors_gdf)
print(f"[dedupe] kept {len(competitors_gdf)}, removed {len(dup_list)} duplicates")

# D-08: GeoJSON persistence — re-runs load from cache, never touch the API
geojson_path = CACHE_DIR / "competitors.geojson"
if geojson_path.exists() and not FORCE_REFRESH:
    competitors_gdf = gpd.read_file(geojson_path)
    print(f"[cache] loaded {len(competitors_gdf)} competitors from {geojson_path.name}")
else:
    competitors_gdf.to_file(geojson_path, driver="GeoJSON")
    print(f"[cache] wrote {len(competitors_gdf)} competitors to {geojson_path.name}")
print(f"[competitors] {len(competitors_gdf)} unique competitors after dedupe + classification")

In [ ]:
# §4.3 Ring assignment via spatial join — which ring is each competitor in?
# Reproject competitors to EPSG:7855 (metric), spatial join with buffer GeoDataFrames
import pandas as pd

competitors_gdf_metric = competitors_gdf.to_crs("EPSG:7855")

# Build per-ring counts table
per_ring_rows = []
for radius in BASE_ASSUMPTIONS["catchment_radii_m"]:
    # Get the buffer for this radius
    ring_gdf = buffers[radius].to_crs("EPSG:7855") if hasattr(buffers[radius], "to_crs") else buffers[radius]
    # Spatial join: which competitors are WITHIN this ring?
    comp_in_ring = gpd.sjoin(competitors_gdf_metric, ring_gdf, predicate="within")

    # Count by category
    counts = comp_in_ring["category"].value_counts().to_dict() if len(comp_in_ring) > 0 else {}
    row = {
        "ring_km": radius // 1000,
        "gp_corporate": counts.get("gp_corporate", 0),
        "gp_independent": counts.get("gp_independent", 0),
        "pharmacy": counts.get("pharmacy", 0),
        "allied_health": counts.get("allied_health", 0),
        "ambiguous": counts.get("ambiguous", 0),
        "exclude": counts.get("exclude", 0),
    }
    row["total"] = sum(v for k, v in row.items() if k != "ring_km")
    per_ring_rows.append(row)

per_ring_counts = pd.DataFrame(per_ring_rows)
print("[ring-assignment] per-ring competitor counts:")
print(per_ring_counts.to_string(index=False))

In [ ]:
# D-11: Manual-review checkpoint — print ambiguous rows inline (no blocking)
# The review is a documented post-run step, not an in-session gate (PIPE-01).
ambiguous = competitors_gdf[competitors_gdf["category"] == "ambiguous"]
if len(ambiguous) > 0:
    print(f"\n⚠ MANUAL REVIEW: {len(ambiguous)} ambiguous competitors flagged.")
    print("Review the table below. If any are misclassified, edit data/cache/competitors.geojson")
    print("manually and re-run from §5.\n")
    review_cols = ["displayName", "formattedAddress", "primaryType", "label"]
    print(ambiguous[review_cols].to_string(index=False))
else:
    print("[review] No ambiguous competitors — all classified successfully.")

## §4.5 Competitor Maps

**Folium** interactive map (inline only, NOT in PDF — D-15): toggleable
ring layers (1/3/5km) + competitor markers coloured by type.

**3 static matplotlib + contextily figures** (one per ring, for PDF — D-15):
competitors coloured by category (GP corporate=red, GP independent=orange,
pharmacy=blue, allied health=green, ambiguous=gray). Same pattern as Phase 2
§2.5 catchment maps.

In [ ]:
# §4.5 Folium interactive competitor map (D-15) — inline only, NOT in PDF
import folium

def make_competitor_map(competitors, site_lat, site_lon, buffers_dict):
    """Interactive competitor map with toggleable ring layers + type colours."""
    m = folium.Map(location=[site_lat, site_lon], zoom_start=13, tiles="CartoDB Positron")

    # Site marker
    folium.Marker(
        [site_lat, site_lon],
        popup="Site: 292-296 Johnston St, Abbotsford",
        icon=folium.Icon(color="red", icon="star", prefix="fa")
    ).add_to(m)

    # Ring layers as FeatureGroups (toggleable)
    ring_colours = {1000: "blue", 3000: "green", 5000: "purple"}
    for radius, colour in ring_colours.items():
        fg = folium.FeatureGroup(name=f"{radius//1000} km ring")
        buf = buffers_dict.get(radius)
        if buf is not None:
            buf_4326 = buf.to_crs("EPSG:4326") if hasattr(buf, "to_crs") else buf
            for _, row in buf_4326.iterrows():
                geom = row.geometry
                if geom.geom_type == "Polygon":
                    folium.Polygon(
                        locations=[(lat, lon) for lon, lat in geom.exterior.coords],
                        color=colour, weight=2, fill=False, dashArray="5"
                    ).add_to(fg)
                elif geom.geom_type == "LineString":
                    folium.PolyLine(
                        locations=[(lat, lon) for lon, lat in geom.coords],
                        color=colour, weight=2, dashArray="5"
                    ).add_to(fg)
        fg.add_to(m)

    # Competitor markers coloured by type
    type_colours = {"gp_corporate": "red", "gp_independent": "orange", "pharmacy": "blue", "allied_health": "green", "ambiguous": "gray"}
    for _, row in competitors.iterrows():
        cat = row.get("category", "ambiguous")
        colour = type_colours.get(cat, "gray")
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=5, color=colour, fill=True, fillOpacity=0.7,
            popup=f"{row.get('displayName', '')} ({cat})"
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    return m

comp_map = make_competitor_map(competitors_gdf, site_lat, site_lon, buffers)
comp_map

In [ ]:
# §4.5 Static matplotlib + contextily figures for PDF (D-15) — one per ring
import matplotlib.pyplot as plt
import contextily as cx

type_colours = {"gp_corporate": "red", "gp_independent": "orange", "pharmacy": "blue", "allied_health": "green", "ambiguous": "gray"}

for radius in BASE_ASSUMPTIONS["catchment_radii_m"]:
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))

    # Plot buffer boundary
    buf = buffers[radius].to_crs("EPSG:7855") if hasattr(buffers[radius], "to_crs") else buffers[radius]
    buf.boundary.plot(ax=ax, color="black", linewidth=1.5, linestyle="--")

    # Plot competitors coloured by category
    for cat, colour in type_colours.items():
        subset = competitors_gdf_metric[competitors_gdf_metric["category"] == cat]
        if len(subset) > 0:
            subset.plot(ax=ax, color=colour, markersize=30, label=cat, alpha=0.7)

    # Add contextily basemap
    cx.add_basemap(ax, crs="EPSG:7855", source=cx.providers.CartoDB.Positron)
    ax.set_title(f"Competitors — {radius//1000} km ring")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

## §4.6 Peer Competitor Table

D-16 — separate from the Phase 2 census-only peer table. This table adds
**GP count, pharmacy count by brand, allied health count, and GP-per-1,000
ratio** per peer. Benchmarked against the VIC average ~117 FTE GPs/100k
(COMP-03). Fills the DEMO-03 placeholder GP/pharmacy columns with real
Places data.

In [ ]:
# §4.6 Peer competitor table (D-16, COMP-03) — GP count, pharmacy by brand, GP-per-1,000
# Separate from the Phase 2 census-only peer table — this adds real Places data.

peer_comp_rows = []
for poa_code in PEER_POSTCODES:
    # Gather all places for this peer across all (type, radius) pairs
    peer_all = []
    for (p_poa, ptype, radius), places in peer_places.items():
        if p_poa == poa_code:
            for p in places:
                dn = p.get("displayName", {})
                name = dn.get("text", "") if isinstance(dn, dict) else str(dn)
                loc = p.get("location", {})
                lat = loc.get("latitude") if isinstance(loc, dict) else None
                lon = loc.get("longitude") if isinstance(loc, dict) else None
                if lat and lon:
                    cat, lbl = classify_place(name, p.get("primaryType", ""), p.get("types", []))
                    peer_all.append({"name": name, "category": cat, "label": lbl})

    # Dedupe on name (peers use centroid, no place.id dedupe across radii)
    seen_names = set()
    unique_peer = []
    for rec in peer_all:
        key = rec["name"].lower().strip()
        if key not in seen_names:
            seen_names.add(key)
            unique_peer.append(rec)

    # Count by category
    gp_corp = sum(1 for r in unique_peer if r["category"] == "gp_corporate")
    gp_indep = sum(1 for r in unique_peer if r["category"] == "gp_independent")
    gp_clinic_count = gp_corp + gp_indep
    pharmacy_count = sum(1 for r in unique_peer if r["category"] == "pharmacy")
    allied_count = sum(1 for r in unique_peer if r["category"] == "allied_health")

    # Pharmacy by brand
    pharmacy_brands = {}
    for r in unique_peer:
        if r["category"] == "pharmacy":
            brand = r["label"]
            pharmacy_brands[brand] = pharmacy_brands.get(brand, 0) + 1

    # GP-per-1,000 using ERP-scaled population from Phase 2
    poa_pop_row = peer_table[peer_table["poa_code"] == poa_code] if "peer_table" in dir() else None
    peer_pop = 0
    if poa_pop_row is not None and not poa_pop_row.empty:
        peer_pop = poa_pop_row.iloc[0].get("Total_P_P_erp", poa_pop_row.iloc[0].get("Total_P_P", 0))

    if peer_pop > 0:
        gp_per_1000 = (gp_clinic_count * BASE_ASSUMPTIONS["avg_fte_per_clinic"]) / (peer_pop / 1000)
    else:
        gp_per_1000 = 0

    peer_comp_rows.append({
        "poa_code": poa_code,
        "gp_clinics": gp_clinic_count,
        "pharmacies": pharmacy_count,
        "allied_health": allied_count,
        "gp_per_1000": round(gp_per_1000, 2),
        "top_pharmacy_brand": max(pharmacy_brands, key=pharmacy_brands.get) if pharmacy_brands else "—",
    })

peer_comp_table = pd.DataFrame(peer_comp_rows)
print("[peer-competitors] peer competitor table:")
print(peer_comp_table.to_string(index=False))
print(f"\n[benchmark] VIC average: {BASE_ASSUMPTIONS['amwac_per_100k']} FTE GPs/100k (AMWAC planning), 117 (VIC actual RACGP 2025)")
print(f"[benchmark] GP-per-1,000 = (gp_clinics × {BASE_ASSUMPTIONS['avg_fte_per_clinic']} FTE/clinic) / (population / 1000)")

## §4.7 Capacity Caveat

> **Caveat (D-12):** Places listings ≠ FTE GPs. The GP capacity estimate
> (computed in §5) spans a **clinic-derived method** (clinic count × 4.0
> FTE/clinic) and a **benchmark-derived method** (AMWAC 110.4/100k ×
> population). The variance IS the caveat — there is no single point
> estimate. The range spans clinic-derived and benchmark-derived estimates.

# §5 Demand Model

v1's "demand model" was a **3-row Random Forest** on peer postcodes —
statistical theatre on a handful of rows that would destroy investor
credibility if probed (Pitfall 14, FEATURES.md Anti-Features). This section
replaces it with **transparent arithmetic**:

    annual_demand = sum(population_band × attendance_rate_band)

Every number is traceable to a cited source. No ML libraries, no regression,
no clustering — just multiplication and addition.

**Key design decisions:**
- The AIHW SA3 age bands are **0-24, 25-44, 45-64, 65+** (Correction 2 —
  NOT 0-14/15-64/65+). Phase 2's ABS 5-year age bands must be aggregated
  to match these 4 bands before multiplying.
- The SA3 20604 MBS data is a **total-attendance cross-check** (computed
  total ≈ SA3 reported total ±15%, D-03).
- The required-market-share is presented in **three framings** (D-13):
  share of total consults, share of unmet demand, share of population.
- The plain-language interpretation labels the share as low/moderate/high
  but does **NOT** issue a go/no-go verdict (D-14 — that's Phase 5 after
  the P&L).

In [ ]:
# §5.1 Aggregate ABS 5-year age bands → AIHW SA3 4 bands (Correction 2)
# AIHW SA3 bands: 0-24, 25-44, 45-64, 65+
# ABS G04 5-year bands: 0-4, 5-9, 10-14, 15-19, 20-24, 25-29, ..., 85+
# Mapping:
#   0-24  = 0-4 + 5-9 + 10-14 + 15-19 + 20-24
#   25-44 = 25-29 + 30-34 + 35-39 + 40-44
#   45-64 = 45-49 + 50-54 + 55-59 + 60-64
#   65+   = 65-69 + 70-74 + 75-79 + 80-84 + 85+
def aggregate_age_bands(g04_age_df):
    """Aggregate ABS 5-year age bands to AIHW SA3 4 bands.
    g04_age_df: DataFrame with age band columns (from Phase 2 §3 G04 fetch).
    Returns dict: {"0-24": pop, "25-44": pop, "45-64": pop, "65+": pop}"""
    band_map = {
        "0-24": ["0_4", "5_9", "10_14", "15_19", "20_24"],
        "25-44": ["25_29", "30_34", "35_39", "40_44"],
        "45-64": ["45_49", "50_54", "55_59", "60_64"],
        "65+": ["65_69", "70_74", "75_79", "80_84", "85_ov"],
    }
    result = {}
    for aihw_band, abs_bands in band_map.items():
        total = 0
        for abs_band in abs_bands:
            # Find matching column(s) — handle naming variations
            cols = [c for c in g04_age_df.columns if abs_band in str(c).replace("-", "_").lower()]
            for c in cols:
                total += g04_age_df[c].sum()
        result[aihw_band] = total
    return result

# Compute per-ring age profiles using POA 3067 G04 data as the age template,
# scaled proportionally to each ring's ERP-scaled population (from Phase 2 §3).
# The catchment is centred on Abbotsford (3067) — POA-level age profile is a
# reasonable proxy for all rings (inner-Melbourne age profiles are similar).
ring_age_profiles = {}
site_g04 = g04_df[g04_df["POA_CODE21"] == "3067"] if "POA_CODE21" in g04_df.columns else g04_df
if len(site_g04) == 0:
    site_g04 = g04_df.iloc[[0]] if len(g04_df) > 0 else g04_df
    print('[demand] ⚠ POA 3067 not in G04 — using first row as age template')

site_age_bands = aggregate_age_bands(site_g04)
site_total_pop = sum(site_age_bands.values())
print(f'[demand] site age bands (POA 3067): {site_age_bands}')

for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    ring_pop = ring_pops_erp.get(r, 0)
    if site_total_pop > 0:
        # Scale age proportions to ring population
        ring_age_profiles[r] = {
            band: (pop / site_total_pop) * ring_pop
            for band, pop in site_age_bands.items()
        }
    else:
        ring_age_profiles[r] = {"0-24": 0, "25-44": 0, "45-64": 0, "65+": 0}
    print(f"[demand] {r//1000}km ring age profile: {ring_age_profiles[r]}")

print("[demand] aggregate_age_bands() defined — maps ABS 5-year → AIHW 4 bands (0-24, 25-44, 45-64, 65+)")

In [ ]:
# §5.2 Compute age-adjusted demand per ring (DEMAND-02, Pitfall 14)
# Transparent arithmetic — no ML. annual_demand = sum(pop_band × rate_band)
def compute_demand(catchment_age_profile, aihw_rates_per_100, ring_pop):
    """Age-adjusted annual GP consult demand for a catchment ring.
    Transparent arithmetic — no ML (Pitfall 14, DEMAND-04).
    catchment_age_profile: dict of age_band → population (AIHW 4 bands)
    aihw_rates_per_100: dict of age_band → services per 100 people (from AIHW)
    ring_pop: total population in the ring (ERP-scaled, from Phase 2)
    Returns: annual GP consults demanded in this ring (float)"""
    total_demand = 0
    for band, pop in catchment_age_profile.items():
        rate = aihw_rates_per_100.get(band, 0)
        total_demand += pop * (rate / 100.0)
    return total_demand

# Compute demand per ring using AIHW rates (from §1.5) and ring age profiles (from §5.1)
ring_demand = {}
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    ring_demand[r] = compute_demand(ring_age_profiles[r], aihw_rates, ring_pops_erp.get(r, 0))
    per_capita = ring_demand[r] / ring_pops_erp.get(r, 1) * 100 if ring_pops_erp.get(r, 0) else 0
    print(f"[demand] {r//1000}km ring | age_adjusted_demand: {ring_demand[r]:.0f} | per_capita_rate: {per_capita:.1f}/100")

print("[demand] compute_demand() defined — sum(pop_band × rate_band), transparent arithmetic (no ML)")

In [ ]:
# §5.3 SA3 20604 MBS total-attendance cross-check (D-03, ±15% tolerance)
# Compare computed total demand against SA3 20604 MBS reported total.
# This validates the demand model against an independent data source.
if mbs_source == "sa3":
    # Sum the latest 4 quarters of SA3 20604 GP non-referred attendances
    # (exact column verified at runtime — look for attendance/count columns)
    numeric_cols = mbs_sa3_df.select_dtypes(include=["number"])
    sa3_total = numeric_cols.sum().sum()  # placeholder — adjust at runtime
    computed_total = sum(ring_demand.values())
    if sa3_total > 0:
        pct_diff = abs(computed_total - sa3_total) / sa3_total * 100
        print(f"[cross-check] computed demand total: {computed_total:.0f}")
        print(f"[cross-check] SA3 20604 MBS total: {sa3_total:.0f}")
        print(f"[cross-check] difference: {pct_diff:.1f}% (tolerance: ±15%)")
        if pct_diff > 15:
            print("[cross-check] ⚠ Difference exceeds 15% — check age band mapping or rates")
        else:
            print("[cross-check] ✓ Within ±15% tolerance")
    else:
        print("[cross-check] ⚠ SA3 total is 0 — check MBS data columns")
else:
    print("[cross-check] ⚠ Skipped — using state fallback, no SA3 total to cross-check")

In [ ]:
# §5.4 GP FTE capacity range (D-10) — two-method range, variance IS the caveat (D-12)
# Method A: clinic count × avg FTE per clinic (4.0 FTE, RACGP+DoH)
# Method B: AMWAC benchmark × population (110.4/100k)
def estimate_gp_capacity_range(clinic_count, ring_pop):
    """D-10: Two-method range. The variance IS the caveat (D-12).
    Method A: clinic count × avg FTE per clinic (4.0 FTE, RACGP+DoH)
    Method B: AMWAC benchmark × population (110.4/100k)
    Returns dict with both methods, range, and caveat string."""
    AVG_FTE_PER_CLINIC = BASE_ASSUMPTIONS["avg_fte_per_clinic"]  # 4.0
    AMWAC_PER_100K = BASE_ASSUMPTIONS["amwac_per_100k"]          # 110.4
    capacity_a = clinic_count * AVG_FTE_PER_CLINIC
    capacity_b = ring_pop * (AMWAC_PER_100K / 100_000)
    return {
        "method_a_clinic_derived": capacity_a,
        "method_b_benchmark_derived": capacity_b,
        "range": (min(capacity_a, capacity_b), max(capacity_a, capacity_b)),
        "caveat": "Places listings ≠ FTE GPs; range spans clinic-derived and benchmark-derived estimates.",
    }

# Compute capacity per ring using per_ring_counts GP clinic count (from §4)
# and ERP-scaled ring population (from §3). Convert FTE to consult capacity:
# existing_consult_capacity = fte_count × gp_fte_consults_per_yr (5500/yr per FTE)
capacity_results = {}
existing_capacity_range = {}
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    ring_row = per_ring_counts[per_ring_counts["ring_km"] == r // 1000]
    if len(ring_row) > 0:
        gp_clinic_count = int(ring_row.iloc[0].get("gp_corporate", 0) + ring_row.iloc[0].get("gp_independent", 0))
    else:
        gp_clinic_count = 0
    ring_pop = ring_pops_erp.get(r, 0)
    cap = estimate_gp_capacity_range(gp_clinic_count, ring_pop)
    capacity_results[r] = cap
    # Convert FTE to consult capacity (5500 consults/yr per FTE)
    consult_cap_a = cap["method_a_clinic_derived"] * BASE_ASSUMPTIONS["gp_fte_consults_per_yr"]
    consult_cap_b = cap["method_b_benchmark_derived"] * BASE_ASSUMPTIONS["gp_fte_consults_per_yr"]
    existing_capacity_range[r] = (min(consult_cap_a, consult_cap_b), max(consult_cap_a, consult_cap_b))
    print(f"[capacity] {r//1000}km ring | GP clinics: {gp_clinic_count} | FTE_a: {cap["method_a_clinic_derived"]:.1f} | FTE_b: {cap["method_b_benchmark_derived"]:.1f} | consult_cap: {existing_capacity_range[r][0]:.0f}–{existing_capacity_range[r][1]:.0f}/yr")

print("[capacity] estimate_gp_capacity_range() defined — two methods (clinic-derived × 4.0 FTE, AMWAC 110.4/100k × pop)")

In [ ]:
# §5.5 Required market share — three framings (D-13, D-14)
# D-13: share_of_total, share_of_unmet, share_of_pop — side-by-side, no ML
# D-14: label_market_share — low/moderate/high, NO go/no-go verdict (Phase 5)
def compute_required_market_share(annual_demand, existing_capacity, clinic_capacity,
                                   ring_pop, consults_per_patient_yr=5.0):
    """D-13: Three framings side-by-side. No ML, pure arithmetic (Pitfall 14).
    annual_demand: age-adjusted GP consults demanded in ring
    existing_capacity: estimated existing GP consult capacity in ring (D-10 range)
    clinic_capacity: 5-FTE clinic annual consult capacity (~27,500/yr)
    ring_pop: total population in ring
    consults_per_patient_yr: ~5.0 (BASE_ASSUMPTIONS)"""
    unmet_demand = max(annual_demand - existing_capacity, 0)
    return {
        "share_of_total":   clinic_capacity / annual_demand * 100 if annual_demand else 0,
        "share_of_unmet":   clinic_capacity / unmet_demand * 100 if unmet_demand else float("inf"),
        "patients_needed":  clinic_capacity / consults_per_patient_yr,
        "share_of_pop":     (clinic_capacity / consults_per_patient_yr) / ring_pop * 100 if ring_pop else 0,
    }

def label_market_share(pct_of_total):
    """D-14: Plain-language label. <5% low, 5-15% moderate, >15% high."""
    thresholds = BASE_ASSUMPTIONS["market_share_thresholds"]
    if pct_of_total < thresholds["low"]:   return "low"
    if pct_of_total < thresholds["high"]:  return "moderate"
    return "high"

# Compute market share for each ring using both capacity methods (D-10 range)
# Clinic capacity: 5 FTE × 5,500/yr = 27,500 consults/yr
clinic_capacity = BASE_ASSUMPTIONS["n_gp_fte"] * BASE_ASSUMPTIONS["gp_fte_consults_per_yr"]  # 27,500/yr
print(f"[market-share] clinic capacity: {clinic_capacity:.0f} consults/yr (5 FTE × 5,500/yr)")

market_share_results = {}
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    cap = capacity_results[r]
    consult_cap_a = cap["method_a_clinic_derived"] * BASE_ASSUMPTIONS["gp_fte_consults_per_yr"]
    consult_cap_b = cap["method_b_benchmark_derived"] * BASE_ASSUMPTIONS["gp_fte_consults_per_yr"]
    ms_a = compute_required_market_share(ring_demand[r], consult_cap_a, clinic_capacity,
                                            ring_pops_erp.get(r, 0), BASE_ASSUMPTIONS["consults_per_patient_yr"])
    ms_b = compute_required_market_share(ring_demand[r], consult_cap_b, clinic_capacity,
                                            ring_pops_erp.get(r, 0), BASE_ASSUMPTIONS["consults_per_patient_yr"])
    # Mid values for the interpretation (average of both methods)
    share_total_mid = (ms_a["share_of_total"] + ms_b["share_of_total"]) / 2
    unmet_a = ms_a["share_of_unmet"]
    unmet_b = ms_b["share_of_unmet"]
    if unmet_a != float('inf') and unmet_b != float('inf'):
        share_unmet_mid = (unmet_a + unmet_b) / 2
    else:
        share_unmet_mid = float('inf')
    market_share_results[r] = {
        "share_of_total_a": ms_a["share_of_total"],
        "share_of_total_b": ms_b["share_of_total"],
        "share_of_total_mid": share_total_mid,
        "share_of_unmet_a": unmet_a,
        "share_of_unmet_b": unmet_b,
        "share_of_unmet_mid": share_unmet_mid,
        "patients_needed": ms_a["patients_needed"],
        "share_of_pop": ms_a["share_of_pop"],
    }
    label = label_market_share(share_total_mid)
    print(f"[market-share] {r//1000}km ring | demand: {ring_demand[r]:.0f} | cap_a: {consult_cap_a:.0f} | cap_b: {consult_cap_b:.0f} | share_total_a: {ms_a["share_of_total"]:.1f}% | share_total_b: {ms_b["share_of_total"]:.1f}% | share_unmet_mid: {share_unmet_mid:.1f}% | share_pop: {ms_a["share_of_pop"]:.1f}% | label: {label}")

print("[market-share] compute_required_market_share() + label_market_share() defined — three framings (D-13), low/moderate/high labels (D-14)")

In [ ]:
# D-14: Plain-language interpretation — NO go/no-go verdict (that's Phase 5)
print("\n" + "=" * 70)
print("REQUIRED MARKET SHARE — PLAIN LANGUAGE SUMMARY")
print("=" * 70)
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    ms = market_share_results[r]
    label = label_market_share(ms["share_of_total_mid"])
    print(f"\n  {r//1000} km ring:")
    print(f"    • {ms['share_of_total_mid']:.1f}% of all GP consults in the ring — {label} share required")
    unmet_str = f"{ms['share_of_unmet_mid']:.1f}%" if ms["share_of_unmet_mid"] != float("inf") else "N/A (no unmet gap)"
    print(f"    • {unmet_str} of the unmet consult gap (can exceed 100%)")
    print(f"    • {ms['patients_needed']:.0f} patients needed = {ms['share_of_pop']:.1f}% of catchment population")
    print(f"    • Clinic capacity: {clinic_capacity:.0f} consults/yr (5 FTE × 5,500/yr)")
    print(f"    • Demand: {ring_demand[r]:.0f} consults/yr (age-adjusted)")
    print(f"    • Existing capacity range: {existing_capacity_range[r][0]:.0f} – {existing_capacity_range[r][1]:.0f} consults/yr")
print("\n  Note: The go/no-go verdict is a Phase 5 output (after P&L + scenarios).")
print("  This section quantifies the required share — it does not judge achievability.")
print("=" * 70)

## §5.7 No-ML Assertion

> **No predictive ML (DEMAND-04, Pitfall 14):** The demand model is
> `sum(population_band × attendance_rate_band)` — pure arithmetic with cited
> rates from AIHW (2026). v1's 3-row Random Forest on peer postcodes was
> statistical theatre on a handful of rows. This section replaces it with
> transparent arithmetic where every input is traceable to a cited source.
> No ML libraries, no regression, no clustering — just multiplication and
> addition.

## Next Steps

**§6 Financial Model (Phase 4)** will build the clinic P&L as a pure function of
`BASE_ASSUMPTIONS`, using the required market share (§5) to derive consult volume →
revenue. The go/no-go verdict is a Phase 5 output after scenarios + sensitivity.

## Next Steps

**§6 Financial Model (Phase 4)** will build the clinic P&L as a pure function of
`BASE_ASSUMPTIONS`, using the required market share (§5) to derive consult volume →
revenue. The go/no-go verdict is a Phase 5 output after scenarios + sensitivity.

## Next Steps

**§6 Financial Model (Phase 4)** will build the clinic P&L as a pure function of
`BASE_ASSUMPTIONS`, using the required market share (§5) to derive consult volume →
revenue. The go/no-go verdict is a Phase 5 output after scenarios + sensitivity.

## Next Steps

**§4 Competitors (Phase 3)** will fetch Google Places (New) data through the
cached session, fill the GP count + pharmacy count columns in the peer
table, and build the demand model using the catchment age profile from §3.

The v1-vs-v2 comparison (§2.4) is now complete with real v2 totals — the
headline teaching moment showing v1's whole-postcode summing inflated
catchment population 2–4×.